In [1]:
from IPython.display import display, HTML
display(HTML('<style>.container { width:80% !important; }</style>'))

In [2]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import matplotlib.pyplot as plt
import math
import urllib.parse
import json
import uuid
import hashlib
import time

In [3]:
pd.set_option('display.max_columns', 500) # To utilise larger part of screen

pd.set_option('display.max_colwidth', None) # To show full cell text

In [4]:
data_path = 'data/cb_20240824'

# Data Import

## Organisations

In [5]:
organisations = pd.read_csv(data_path + '/organizations.csv')[['uuid', 'name', 'country_code', 'region', 'city', 'status', 'short_description', 'category_list', 'category_groups_list', 'num_funding_rounds', 'total_funding_usd', 'founded_on', 'logo_url']]
organisations = organisations.rename(columns = {'uuid': 'org_uuid', 'name': 'org_name', 'country_code': 'org_country_code', 'region': 'org_region', 'city': 'org_city', 'logo_url': 'org_logo_url'})

organisations = organisations.assign(founded_on = pd.to_datetime(organisations['founded_on'], errors = 'coerce'))

organisations['total_funding_usd'] = organisations['total_funding_usd'].astype('Int64')
organisations['num_funding_rounds'] = organisations['num_funding_rounds'].astype('Int64')

organisations.head()

,org_uuid,org_name,org_country_code,org_region,org_city,status,short_description,category_list,category_groups_list,num_funding_rounds,total_funding_usd,founded_on,org_logo_url
0,e1393508-30ea-8a36-3f96-dd3226033abd,Wetpaint,USA,New York,New York,acquired,Wetpaint offers an online social publishing platform that helps digital publishers grow their customer base.,"Publishing,Social Media,Social Media Management","Content and Publishing,Internet Services,Media and Entertainment,Sales and Marketing",3,39750000,2005-06-01,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180177/2036b3394a37152e0ff69f27c71bc883.jpg
1,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,Zoho,USA,California,Pleasanton,operating,"Zoho offers a suite of business, collaboration, and productivity applications.","Cloud Computing,Collaboration,Developer Tools,Enterprise Software,Information Services,Information Technology,Network Security,Project Management,Software,Web Apps","Administrative Services,Apps,Information Technology,Internet Services,Other,Privacy and Security,Software",<NA>,<NA>,1996-03-17,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180181/f8aaab73f17af0296eba5deda7a5b95b.png
2,5f2b40b8-d1b3-d323-d81a-b7a8e89553d0,Digg,USA,New York,New York,acquired,"Digg Inc. operates a website that enables its users to find, read, and share the most interesting and talked about stories on the internet.","Internet,Social Media,Social Network","Internet Services,Media and Entertainment",6,49000000,2004-10-11,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180182/e77f8f561153ffb45a9ffd538978380d.jpg
3,f4d5ab44-058b-298b-ea81-380e6e9a8eec,Omidyar Network,USA,California,Redwood City,operating,Omidyar Network is an investment firm.,"Enterprise Software,Financial Services,Venture Capital","Financial Services,Lending and Investments,Software",<NA>,<NA>,2004-01-01,https://images.crunchbase.com/image/upload/t_cb-default-original/v1491524876/tjg7440r2uc0s5neswkp.png
4,df662812-7f97-0b43-9d3e-12f64f504fbb,Meta,USA,California,Menlo Park,ipo,"Meta is a social technology company that enables people to connect, find communities, and grow businesses.","Artificial Intelligence (AI),Augmented Reality,Metaverse,Social Media,Social Network,Virtual Reality","Artificial Intelligence (AI),Data and Analytics,Hardware,Internet Services,Media and Entertainment,Science and Engineering,Software",14,24607817488,2004-02-04,https://images.crunchbase.com/image/upload/t_cb-default-original/whm4ed1rrc8skbdi3biv


## People

In [6]:
people = pd.read_csv(data_path + '/people.csv')[['uuid', 'name', 'country_code', 'region', 'city', 'featured_job_organization_uuid', 'featured_job_title', 'logo_url']]
people = people.rename(columns = {'uuid': 'person_uuid', 'name': 'person_name', 'country_code': 'person_country_code', 'region': 'person_region', 'city': 'person_city', 'featured_job_organization_uuid': 'featured_job_org_uuid', 'logo_url': 'person_logo_url'})

people.head()

,person_uuid,person_name,person_country_code,person_region,person_city,featured_job_org_uuid,featured_job_title,person_logo_url
0,ed13cd36-fe2b-3707-197b-0c2d56e37a71,Ben Elowitz,USA,Washington,Seattle,1d845b32-7d80-47af-957d-78ccbeeaefb6,Co-Founder,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180224/56303d2f4b99dd1dcc8abf17ba3fd8bf.jpg
1,5ceca97b-493c-1446-6249-5aaa33464763,Kevin Flaherty,USA,Washington,Mercer Island,789e5e4d-0c90-d06e-92a0-b800b461c3da,Team Member,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180225/a307ba697e0f042623f05f35477bf495.jpg
2,9f99a98a-aa97-b30b-0d36-db67c1d277e0,Raju Vegesna,USA,California,San Francisco,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,Chief Evangelist,https://images.crunchbase.com/image/upload/t_cb-default-original/oaxfoww3m0u0lwdovzv3
3,6e1bca72-a865-b518-b305-31214ce2d1b0,Ian Wenig,NaN,NaN,NaN,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,VP Business Development,https://images.crunchbase.com/image/upload/t_cb-default-original/v1442309935/yiajbpfbxyopc5l4zs4t.png
4,3b598c59-7b6c-2d48-763c-da55bca77035,Owen Byrne,USA,California,Mountain View,NaN,NaN,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180229/eb6f898475c0bef4c624ed44c9add19a.jpg


In [7]:
# Manual additions and changes

new_records = pd.DataFrame([
    # ---------------- PayPal ----------------
    {
        'person_uuid': '0191214a-e133-78ba-84ce-3777ffbd4b57',
        'person_name': 'Jason Portnoy',
        'person_country_code': 'USA',
        'person_region': 'Utah',
        'person_city': 'Park City',
        'featured_job_org_uuid': '5c26c58f-6d80-43e8-8022-24c2d769af0a',
        'featured_job_title': 'Founder and Managing Partner',
        'person_logo_url': 'https://media.licdn.com/dms/image/C5603AQEjs4rrrTg1GQ/profile-displayphoto-shrink_800_800/0/1645129071256?e=1728518400&v=beta&t=g_0vWA6xwzXg9ZLI9qv6sq5oo2g0FrhSwS9O2LKOWYQ'
    },
    # ---------------- iZettle ----------------
    {
        'person_uuid': '019184cd-cee7-73cd-ae18-a4a414f4035f',
        'person_name': 'Klas Johansson',
        'person_country_code': 'SWE',
        'person_region': 'Stockholms Lan',
        'person_city': 'Stockholm',
        'featured_job_org_uuid': '2c4bdca3-1244-4d89-8912-931db7573343',
        'featured_job_title': 'Co-founder and CEO',
        'person_logo_url': 'https://media.licdn.com/dms/image/v2/D4D03AQGO6DxJ-T0Uow/profile-displayphoto-shrink_800_800/profile-displayphoto-shrink_800_800/0/1695294373032?e=1729728000&v=beta&t=PuN58kiAEnUF7prgophrS62r5dZPBIiJxf9YiuxayxA'
    },
        {
        'person_uuid': '019184cf-f87f-7c70-b207-004b1359a76f',
        'person_name': 'Ruben Flam',
        'person_country_code': 'SWE',
        'person_region': 'Stockholms Lan',
        'person_city': 'Stockholm',
        'featured_job_org_uuid': '2c4bdca3-1244-4d89-8912-931db7573343',
        'featured_job_title': 'Co-founder and CEO',
        'person_logo_url': 'https://media.licdn.com/dms/image/v2/D4E03AQEGjAlG5oTK6A/profile-displayphoto-shrink_800_800/profile-displayphoto-shrink_800_800/0/1707931042804?e=1729728000&v=beta&t=O3-mnx8xuhogBcaJlv-xGWkyaYRcnc4PiFRNv_R4eEA'
    },
])

people.loc[people['person_uuid'] == '5c0a828d-ea80-44a7-823f-1389786c38ef', 'person_name'] = 'Fredrik Jung Abbou'

people = pd.concat([people, new_records], ignore_index = True)

## Jobs

In [8]:
jobs = pd.read_csv(data_path + '/jobs.csv')[['uuid', 'person_uuid', 'org_uuid', 'started_on', 'ended_on', 'is_current', 'title', 'job_type']]
jobs = jobs.rename(columns = {'uuid': 'job_uuid', 'title': 'job_title'})

jobs = jobs.assign(started_on = pd.to_datetime(jobs['started_on'], errors = 'coerce'))
jobs = jobs.assign(ended_on = pd.to_datetime(jobs['ended_on'], errors = 'coerce'))

jobs.head()

,job_uuid,person_uuid,org_uuid,started_on,ended_on,is_current,job_title,job_type
0,697b6934-fc1f-9d63-cfb2-1a10759b378e,ed13cd36-fe2b-3707-197b-0c2d56e37a71,e1393508-30ea-8a36-3f96-dd3226033abd,2005-10-01,2014-06-01,False,Co-Founder and CEO,executive
1,b1de3765-442e-b556-9304-551c2a055901,5ceca97b-493c-1446-6249-5aaa33464763,e1393508-30ea-8a36-3f96-dd3226033abd,NaT,NaT,False,VP Marketing,executive
2,1319cd30-f5e8-c700-0af6-64029c6f7124,9f99a98a-aa97-b30b-0d36-db67c1d277e0,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,2000-11-01,NaT,True,Chief Evangelist,employee
3,27a252de-1ea8-c620-b2d4-5b889fa9b40f,6e1bca72-a865-b518-b305-31214ce2d1b0,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,2006-03-01,NaT,True,VP Business Development,executive
4,5a802a79-229f-44ae-0aba-db330f10b67a,c92a1f00-8c19-bf2e-0f28-dbbd383dc968,5f2b40b8-d1b3-d323-d81a-b7a8e89553d0,2005-07-01,2010-04-05,False,CEO,executive


In [9]:
# Merging the 2 eBay entities
jobs.loc[jobs['org_uuid'] == 'e722cbfc-7594-41b2-b5a9-2b16516f47c7', 'org_uuid'] = 'e56b0ceb-bb30-bbec-805e-d5dc7412dcb1'

----------- Support Start -----------

In [10]:
# Support for adding information
jobs_orgs_search = pd.merge(
    jobs,
    organisations,
    how = 'left',
    on = 'org_uuid'
)[['job_uuid', 'person_uuid', 'org_uuid', 'org_name', 'started_on', 'ended_on', 'job_title', 'job_type']]

In [102]:
people.loc[people['person_name'] == 'Yvonne Chen']

,person_uuid,person_name,person_country_code,person_region,person_city,featured_job_org_uuid,featured_job_title,person_logo_url
383537,3522c383-8cfa-abc3-fb50-3d81debcb635,Yvonne Chen,TWN,T'ai-pei,Taipei,ca01f2b5-d280-cae7-8603-81609771c485,Venture Partner,https://images.crunchbase.com/image/upload/t_cb-default-original/q8zbmdo02x0ae8xe1h0l
476310,b36c150b-968b-884d-4109-d6a7ed0e3ae7,Yvonne Chen,USA,California,San Francisco,fcb75056-1717-697c-fdd2-c2643146e9dc,VP of Marketing,https://images.crunchbase.com/image/upload/t_cb-default-original/v1464399138/amawb6tqfq9upt95xsou.png
1055570,4313f140-ce82-483f-a6d3-c63b8b60a70f,Yvonne Chen,NaN,NaN,NaN,NaN,NaN,NaN


In [148]:
# Support for adding information
jobs_orgs_search.loc[
    (jobs_orgs_search['person_uuid'] == people.loc[people['person_name'] == 'Paul Ollinger']['person_uuid'].values[0])
    & (jobs_orgs_search['org_uuid'] == organisations.loc[organisations['org_name'] == 'Meta']['org_uuid'].values[0])
]

,job_uuid,person_uuid,org_uuid,org_name,started_on,ended_on,job_title,job_type
231725,ac1488ad-a47c-0087-71e1-3b5515318467,df3196e3-cd74-3dc5-78be-41b05be09243,df662812-7f97-0b43-9d3e-12f64f504fbb,Meta,2007-05-27,2011-09-16,"VP of Sales, West Region",executive


In [63]:
alumni_master.loc[(alumni_master['org_name_target'] == 'Meta') & ~(alumni_master['person_name'].isin(['Magnus Nilsson']))].sort_values(by = 'person_uuid')#.tail(50)#.drop_duplicates(subset = ['person_uuid'])

,person_uuid,org_uuid_target,started_on_target,ended_on_target,job_title_target,job_type_target,org_name_target,org_country_code_target,org_city_target,founded_on_target,person_name,person_logo_url,org_uuid_subsequent,started_on_subsequent,ended_on_subsequent,job_title_subsequent,relation_type,org_name_subsequent,org_country_code_subsequent,org_city_subsequent,founded_on_subsequent,short_description_subsequent,total_funding_usd_subsequent,org_logo_url_subsequent,acquirer_uuid_subsequent,exit_type_subsequent,acquirer_name_subsequent,exit_date_subsequent,acquisition_type_subsequent,exit_valuation_subsequent,record_uuid
8741520,002c9bce-60d3-4889-a6d4-52a01f9e3a3e,df662812-7f97-0b43-9d3e-12f64f504fbb,2008-01-01,2017-02-01,Director of International Business Development & Mobile Partnerships - LATAM,executive,Meta,USA,Menlo Park,2004-02-04,Laura Gonzalez-Estefani,https://images.crunchbase.com/image/upload/t_cb-default-original/qfnkypjlzw9yk56mxtoz,3871e66d-77bc-49c1-b2eb-05167b19ac50,2023-04-27,NaT,investor,investor,HumanForest,GBR,London,2020-01-01,"HumanForest is a dockless, shared, and ad-supported e-bike service company.",28523918,https://images.crunchbase.com/image/upload/t_cb-default-original/qcbove4jldkh9kpovs7u,NaN,NaN,NaN,NaT,NaN,<NA>,03c2bf3a-82d7-4c95-ac90-0b4ce4d46509
8741519,002c9bce-60d3-4889-a6d4-52a01f9e3a3e,df662812-7f97-0b43-9d3e-12f64f504fbb,2008-01-01,2017-02-01,Director of International Business Development & Mobile Partnerships - LATAM,executive,Meta,USA,Menlo Park,2004-02-04,Laura Gonzalez-Estefani,https://images.crunchbase.com/image/upload/t_cb-default-original/qfnkypjlzw9yk56mxtoz,d0b299bb-ab11-c8b1-0511-37202f9c8e18,2017-03-09,NaT,investor,investor,WOOM,ESP,Madrid,2015-01-01,WOOM is Data science for Women's and Reproductive Health,4400703,https://images.crunchbase.com/image/upload/t_cb-default-original/qbqqchssqxoazni6khqw,d6f17561-7a9d-4300-ad00-8f411c07ddd4,acquisition,Apricity,2022-05-30,acquisition,<NA>,750acc1f-3012-495d-9011-d532df3ffd08
8741515,002c9bce-60d3-4889-a6d4-52a01f9e3a3e,df662812-7f97-0b43-9d3e-12f64f504fbb,2008-01-01,2017-02-01,Director of International Business Development & Mobile Partnerships - LATAM,executive,Meta,USA,Menlo Park,2004-02-04,Laura Gonzalez-Estefani,https://images.crunchbase.com/image/upload/t_cb-default-original/qfnkypjlzw9yk56mxtoz,31f38b11-4f52-8301-dc9c-4f0c7e80af32,2017-03-01,NaT,"Founder, CEO & General Partner",executive,TheVentureCity,USA,Miami,2017-01-01,TheVentureCity is a global early-stage venture fund that invests in product-centric startups.,<NA>,https://images.crunchbase.com/image/upload/t_cb-default-original/bj5ck1wwshj4zlmaiycu,NaN,NaN,NaN,NaT,NaN,<NA>,65f80edc-2363-41f3-a744-cbe24bbd7b2d
8741518,002c9bce-60d3-4889-a6d4-52a01f9e3a3e,df662812-7f97-0b43-9d3e-12f64f504fbb,2008-01-01,2017-02-01,Director of International Business Development & Mobile Partnerships - LATAM,executive,Meta,USA,Menlo Park,2004-02-04,Laura Gonzalez-Estefani,https://images.crunchbase.com/image/upload/t_cb-default-original/qfnkypjlzw9yk56mxtoz,91a7b787-ac6e-4baa-8397-44d00eaa03f9,2019-09-01,NaT,Advisor,advisor,Erudit,USA,Miami,2020-01-01,"Erudit applies the latest technologies so companies get reliable, timely People Analytics for data-powered decisions.",15314339,https://images.crunchbase.com/image/upload/t_cb-default-original/ibykirybo4jmjf7xzrpn,NaN,NaN,NaN,NaT,NaN,<NA>,0ac5e480-d96e-4c2c-956a-21752a3c614e
990467,03c01b54-10a2-c163-5fc2-d961be781b29,df662812-7f97-0b43-9d3e-12f64f504fbb,2007-05-01,2008-11-01,Engineering Manager,executive,Meta,USA,Menlo Park,2004-02-04,Justin Rosenstein,https://images.crunchbase.com/image/upload/t_cb-default-original/v1495344696/mx4ypllnpbrrs2hdldub.png,33465568-50ca-6b19-75ee-9d91e7db3413,2017-06-12,NaT,Adviser,advisor,Bancor,CHE,Zug,2016-08-21,Bancor is crypto company protocol for the creation of Smart Tokens that touts a decentralized exchange service.,152310000,https://images.crunchbase.com/image/upload/t_cb-default-original/v1506236723/sihrjelq7

In [68]:
alumni_master.loc[(alumni_master['org_name_target'] == 'Meta')].drop_duplicates(subset = ['person_uuid', 'job_title_target'])[['person_name', 'job_title_target', 'started_on_target', 'ended_on_target']]

,person_name,job_title_target,started_on_target,ended_on_target
6715662,Mike Vernal,"Vice President of Search, Local, and Developer products",2008-01-01,2016-04-01
6695129,Cameron Marlow,"Manager, Data Science",2007-11-01,2013-12-01
4946182,Nick Heyman,Director of Operations,2005-04-01,2007-01-01
5172958,Edward Suh,Software Development Intern,2008-01-01,2008-01-01
4630234,TS Ramakrishnan,VP Product Engineering,2005-01-01,2006-01-01
4526414,Peter Yewell,Sales Director - Partnerships and Marketing,2006-04-01,2012-04-01
4539962,John Hegeman,Director,2007-01-01,2013-01-01
4558063,Benjamin Ling,"Director, Platform",2007-10-01,2008-08-01
4558004,Paul Ollinger,"VP of Sales, West Region",2007-05-27,2011-09-16
5772062,Doug Hirsch,VP Product,2005-01-01,2006-01-01


In [61]:
len(alumni_master.loc[(alumni_master['org_name_target'] == 'Meta')]['person_uuid'].unique())

50

In [147]:
alumni_master.loc[(alumni_master['org_name_target'] == 'Meta') & (alumni_master['person_name'] == 'Paul Ollinger')]

,record_uuid,person_uuid,org_uuid_target,started_on_target,ended_on_target,job_title_target,job_type_target,org_name_target,org_country_code_target,org_city_target,founded_on_target,person_name,person_logo_url,org_uuid_subsequent,started_on_subsequent,ended_on_subsequent,job_title_subsequent,relation_type,org_name_subsequent,org_country_code_subsequent,org_city_subsequent,founded_on_subsequent,short_description_subsequent,total_funding_usd_subsequent,org_logo_url_subsequent,acquirer_uuid_subsequent,exit_type_subsequent,acquirer_name_subsequent,exit_date_subsequent,acquisition_type_subsequent,exit_valuation_subsequent
4558018,f4a20966-3ac4-4262-87fe-6e99abeaef66,7089385d-5c8e-e2ae-365e-e89d04bee929,554b7e5e-e5e0-1a4c-8dab-084840575e0c,2007-05-27,2011-09-16,"VP of Sales, West Region",executive,Meta,USA,Menlo Park,2004-02-04,Paul Ollinger,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397184501/cda978b3566d739f7a13fc81d7726fb8.jpg,cfd50418-3f7f-76c8-7fbc-9d276020512c,2013-07-01,2014-10-01,President,executive,SHIFT,USA,Santa Monica,2010-01-01,SHIFT is a leading cross-network social advertising platform for brands and agencies.,14000000,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397194979/163cc24ad7ec02b12fa748e14c7c8148.png,3e3c089d-01d4-7847-2dde-72c668000cd3,acquisition,Brand Networks,2015-05-14,acquisition,50000000
4558020,ef8d9162-2474-4d0d-bf55-74a95354c1c4,7089385d-5c8e-e2ae-365e-e89d04bee929,554b7e5e-e5e0-1a4c-8dab-084840575e0c,2007-05-27,2011-09-16,"VP of Sales, West Region",executive,Meta,USA,Menlo Park,2004-02-04,Paul Ollinger,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397184501/cda978b3566d739f7a13fc81d7726fb8.jpg,adb734c1-2847-c9f1-b9fb-c6cba620870f,2013-02-01,2013-06-26,Advisor,advisor,LeanKit,USA,Franklin,2009-05-01,LeanKit is an enterprise process and work management software that helps enterprises visualize work and optimize processes.,28599992,https://images.crunchbase.com/image/upload/t_cb-default-original/ijrc91mquh7ne5mn96xi,469cf51c-afa1-6a95-8f10-20a640a7c0b9,acquisition,Planview,2017-12-11,acquisition,<NA>
4558021,bb47a459-4a4d-4685-95fe-8671c2ec08a3,7089385d-5c8e-e2ae-365e-e89d04bee929,554b7e5e-e5e0-1a4c-8dab-084840575e0c,2007-05-27,2011-09-16,"VP of Sales, West Region",executive,Meta,USA,Menlo Park,2004-02-04,Paul Ollinger,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397184501/cda978b3566d739f7a13fc81d7726fb8.jpg,1147a90e-5a6e-925c-a80d-69c49db69a24,2014-12-01,NaT,Advisor,advisor,ATOMIZED,USA,Atlanta,2013-12-15,ATOMIZED is an omnichannel visual content calendar designed with marketers in mind.,5955000,https://images.crunchbase.com/image/upload/t_cb-default-original/tphpye6vs35gz3330o8s,cfe64f10-8e4d-9eab-6c53-c0096dc45b09,acquisition,OPAL,2019-08-01,NaN,<NA>


In [13]:
jobs.loc[
    (jobs['person_uuid'] == people.loc[people['person_name'] == 'Omid Kordestani']['person_uuid'].values[0])
    & (jobs['org_uuid'] == 'cc68526b-b2d7-4f7f-cfa7-d93b23716027')
]

,job_uuid,person_uuid,org_uuid,started_on,ended_on,is_current,job_title,job_type
436345,9b1fed49-bf42-75e0-a652-0188abf61f79,b42cd994-c370-73d6-1702-1f590da4f71b,cc68526b-b2d7-4f7f-cfa7-d93b23716027,1995-01-01,1999-01-01,False,Vice President of Business Development,executive


In [14]:
jobs.loc[
    (jobs['person_uuid'] == people.loc[people['person_name'] == 'Sven Perkmann']['person_uuid'].values[0])
]

,job_uuid,person_uuid,org_uuid,started_on,ended_on,is_current,job_title,job_type
1105498,3c652a06-73aa-4559-a3b5-82139ca0cbc4,df3fcb4a-a555-4a3b-8832-74f57933710a,6f8f114c-8df6-49de-a301-62242a78efa1,2017-03-01,NaT,True,CTO and Co-Founder,executive


In [15]:
organisations.loc[organisations['org_name'] == 'Klarna']

,org_uuid,org_name,org_country_code,org_region,org_city,status,short_description,category_list,category_groups_list,num_funding_rounds,total_funding_usd,founded_on,org_logo_url
35226,2cc3a5de-2303-aa00-cd1a-50bd96420392,Klarna,SWE,Stockholms Lan,Stockholm,operating,Klarna is an e-commerce payment solutions platform for merchants and shoppers.,"E-Commerce,FinTech,Payments,Shopping","Commerce and Shopping,Financial Services,Payments",32,4548376760,2005-02-01,https://images.crunchbase.com/image/upload/t_cb-default-original/i7yjhhitbze4kylwmbfy


----------- Support End -----------

In [10]:
# Manual additions and changes

jobs = jobs.loc[~jobs['job_uuid'].isin([
    # ---------------- Kry ----------------
    '53c1808b-2336-ed4c-ceb0-6c0c6e3ac073', # Sabina Wizander, Director business operations by CSO
    'ee2ecaa4-5089-4a98-ac54-fcab77e8d43c', # Sabina Wizander, CCO by CSO
])]

# employee --> executive
jobs.loc[jobs['job_uuid'].isin([
    # ---------------- PayPal ----------------
    '414abaca-8832-12d1-2c1b-8443cb4c7e37', # Reid Hoffman, Executive VP
    'f2da8e78-ad0d-8385-7e09-6ad9ac86951b', # Max Levchin, Co-founder & CTO
    '3d1eb719-9c60-7813-4f06-7ee07aa5ccd1', # Keith Rabois, Executive VP
    # ---------------- LinkedIn ----------------
    '5cf7c02c-2b7f-5ffc-3b0a-8e75d1b786c5', # Michael Gamson, SVP global solutions
    '8f541146-d14a-e8b6-f22a-04b4a3cc100e', # Konstantin Guericke, Co-founder and VP Marketing
    # ---------------- Skype ----------------
    'c5edc43e-b679-7d06-fd4b-a4c977076b3d', # Ahti Heinla, Founding engineer, Chief architect
    'e5dc6ffe-34b7-1e42-aeb8-df513c44d8ec', # Eileen Burbidge, Product development director
    # ---------------- eBay ----------------
    '57fa88d6-d43f-4eb7-ae25-70fdad9d8f24', # Alexander Samwer, Founding partner Pelion Green Future
    # ---------------- Spotify ----------------
    '9efd68ab-7ff2-4ac0-900b-b23784962e5b', # Andreas Blixt, Vice President of Engineering
    'b8613598-b40f-4cb8-ba94-c0effef82a1d', # Nevyn Bengtsson, Co-founder, CEO, and Code Poet
    'f85b172b-d2f9-4308-baa5-950ca5eb145d', # Andreas Ehn, Partner
    'a9d05523-640e-41c1-b193-4ff9a5f85abe', #Pascal De Mul, Head of International Growth
    'c54daac7-1aa5-205b-2155-1d7dd28a1224', # Henrik Torstensson, Head of Premium Sales
]), 'job_type'] = 'executive'

# executive --> employee
jobs.loc[jobs['job_uuid'].isin([
    # ---------------- PayPal ----------------
    'a42cbc0b-25db-7f10-89e6-73f251588e06', # Michael Blum, Senior Manager, International
    'f352aa4d-7806-ad45-65ae-5308bdd97a9f', # Ryan Downs, VP Operations (only 6 months)
    '17865d6d-6a4c-3c11-e316-c3f3e69ae0f7', # Mark Goldenson, Product Manager
    '01007679-1e44-3c07-7d77-544bf713c373', # Patrick Breitenbach, Product Manager
    '8ff08b7e-cf9d-6816-847c-f0f2af869942', # Ken Brownfield, Manager, Systems Integration
    '18f9c155-930f-2728-ff7c-b928db649abb', # Kevin Freedman, Senior Manager, PayPal FP&A
    '65c25952-d38b-cb2f-eca3-54de8cd8f39f', # Scott Braunstein, VP of European Development
    # ---------------- LinkedIn ----------------
    '4ecf3342-7243-d2d8-d7c5-d885e821d96d', # Babak Rabiee Relationship Manager
    '6dc16f92-8d98-4ca1-8072-ac05e726110e', # Peter Thiel, Investor
    '54acbf93-8108-9c0e-2ca3-2b660da5b6d6', # Mrinal Desai, Business development manager
    '335cac5e-f5e0-228c-2c65-7ed991959337', # Nalini Arulmozhi, QA Manager
    'dac0a84e-3f2f-f369-5a8d-70bd20b9b05e', # Eishay Smith, Principal engineer
    # ---------------- Skype ----------------
    '89b720d2-db0c-a1f7-294d-f94a99c5b256', # Andy Chen, Strategic partnershop and ODM ecosystem consultant
    'b3445d3f-b126-6e36-3dc3-05b15e0ce5f6', # Janno Teelem, Mac product engineering manager
    'a832596d-9e4b-6b78-3dd1-99f74c884fed', # Kaili Kleemeier, Payment operations manager
    'cad8e9ad-0ad5-663c-8558-c50195211f14', # Kaili Kleemeier, Operations manager
    'caf31a90-38ab-6ef2-60c8-5e79f9db5efb', # Kaili Kleemeier, Operations change management
    '4cec50e8-6e68-45e6-8a60-071e313b90eb', # Kristo Magi, Senior web QA engineer
    # ---------------- eBay ----------------
    '5d145a44-f313-213b-4f76-eed5c83a5e97', # Eric Moriarty, Senior category manager
    'a3f2713e-7b64-1574-70fa-89bb7f050d08', # Dave Heilmann, Product manager
    'adb42e22-18c1-48be-40a7-33a4f60da8d9', # Faith Sedlin, Product manager
    '5ae995c9-8d3d-4351-9486-5b8ffcc9b367', # Rob Ratterman, Special projects
    # ---------------- Netscape ----------------
    '7a877284-1646-c913-4962-519bcb7fa84f', # Bala Guthy, Website Dev Manager
    '974c54f5-8a07-f155-01df-65cb3347abdd', # Christopher Walton, Product Manager
    "522f1843-b04e-7ac2-9243-2c131216253d", # Richard Shusterman, Development Manager
    "c51d0951-ebe2-362b-1486-0b7f914967b3", # Yuan Huntington Weigel, Director, Netscape / America Online
    "f9212db9-b696-6de3-20c0-b2c84edda49d", # Ray Rike, VP Worldwide sales and Marketing
    "1f358901-2a95-629e-1386-0e37c3d1df5c", # Dan Kuokka, Sr Engineering Manager
    "ec6141d1-9be7-0bf3-c1b0-49e7c0a97ece", # Stephen Shoaff, Sr Product Manager
    "add9a1bb-9379-e3c9-8c46-a8779cebcbb6", # Suresh Patel, Director, Regional Business Development EMEA
    "bbf6bbef-ebb6-464b-603f-8526ac212fca", # David Kandasamy, Manager
    "004ee3b5-29ec-4271-e116-4513e383c4c4", # Andres Espineira, Director Product Marketing
    "3af5ff9d-fc7a-311b-e5e4-a34aaba8e0bd", # Andy Spillane, Sr Principal Consultant
    "ae2479ec-261b-fc35-84a7-390f343c4ecc", # Vishy Ramachandran, Technical Manager
    "ecc977fc-e08e-f747-8036-40efc45b8b46", # Warren Nagatani, Manager
    "9e397f4f-4198-190e-0b89-79b883120ff6", # Keng Lim, VP
    "09197c88-190a-0ff6-48be-7f1d45c56908", # David Weiden, VP
    "f2c67bd5-14bc-fa48-a91f-db5e1c00600c", # Ammiel Kamon, VP
    "0aeb43ca-d143-81fb-dd78-9ed7628d7e01", # Suzanne Usiskin, Group Product Marketing Manager
    "b2bb6562-6a6c-922c-ad33-6d1797b2d7a8", # James Barksdale, President and CEO (only 2 executive subsequent)
    "602b2d30-1ec8-3cd2-01b8-a23ddc592626", # Jon Mittelhauser, Founding Engineer (only 2 executive subsequent)
    "5868207a-9482-d097-c0b9-28192d918a86", # Jason Rosenthal, Director of Product Management
    "38918c7e-bffc-4683-8a93-e2b457e6d841", # Jason Rosenthal, Director
    "9b8e67f5-fb9f-434c-a45f-703138431a3f", # Tony Fernandes, Senior Manager
    "96a9befe-1ba3-9ca8-74ea-02fbe83cba21", # Didier Bench, General Manager & VP EMEA (only 3 subsequent)
    "04b388b6-acbf-ce3f-1e5d-8c99bd3dfa4e", # John Bruggeman, VP of Strategic Planning (only 4 subsequent)
    "54acddaf-4c62-cdf2-84a5-f01d00920612", # David Pann, Sr Director
    "41036c3b-3dc3-f29e-34f9-c4be810c8779", # Dave Keefer, Engineering Manager
    "f31c5987-dade-815e-2c92-b82256a136a8", # Gary Nakamura, Manager
    "bce04ce7-79d0-15dd-5053-f0c16b320f54", # Alan Louie, Director Business Development and OEM Sales
    "5312a006-62b7-c275-e112-258d6436ed13", # Alex Edelstein, Director of Product Management
    "f1d1d1b4-8e2a-aa54-91d0-cdab3578078e", # Albert Gouyet, Director, Product Management & Marketing
    # ---------------- Meta ----------------
    'aade8497-c915-e2eb-3faa-c896833bea86', # Steven Trieu, Director of Finance & Business Operations
    '2d60c4ac-6418-5d86-6259-63eb2c79a095', # Ezra Callahan, Manager of International Communications
    '8a9acd85-6a54-4f99-8318-e57eb6b49070', # Laura Gonzalez-Estefani, Director of International Business Development & Mobile Partnerships - LATAM
    '40921293-3bd8-71df-2493-dad84cb6b296', # Yvonne Chen, Account Mangaer
    '59886ba8-f836-11b3-a47a-e458e2aa1394', # Matt Hicks, Manager, Corporate Communications
    '82108b6b-c37c-3e68-b08a-fd4dc2b2a7cf', # Peter Yewell, Sales Director
    '1fdae444-3557-466f-9a0a-9ada67a875de', # Patrice Carrel, SEO Consultant
    'b397fad9-db7f-ed9c-b366-b8ff94dd6f8d', # Bob Trahan, Director of Engineering and Culture
    '27f2dc9f-34f3-8159-a5dc-0b200e970e02', # Rebekah Cox, Product Design Lead
    '65e84e10-60b6-1693-3967-4e9031372b53', # John Hegeman, Director
    '75d9a43f-5d73-7f76-0020-613ffe97bb3d', # Nick Heyman, Director of Operations
    'f84cb66d-f1c2-d86f-c011-3f28f2bd85e3', # Luke Shepard, Software Engineer / Manager
    'af42963e-b0c7-67cb-0fdd-1af8007a0928', # Ami Vora, VP / Director, Ads
    'e65c48f5-706e-a12f-4303-1741def33a1a', # Jason Sobel, Director, Engineering
    '9f3803ed-7c08-f1f4-fa78-7b33d7611a75', # Katie Geminder, Director, User Experience and Design
    'fc453c0f-0cd5-9f0f-53ef-693ba498d3f2', # Meagan Marks, Principal
    'd46776f1-df4b-10c4-b91c-bb7cab85d94a', # Andrew Bosworth, VP Engineering, Ads and Pages
    '40bde8e2-bd3b-441f-95fc-de6bc81ce866', # Paul Jeffries, Head of Platform Developer Operations & Support
    'ac1488ad-a47c-0087-71e1-3b5515318467', # Paul Ollinger, VP of Sales, West Region
    '03ee6f41-d2f0-8b5b-50e7-59f9f90de49a', # Naomi Gleit, Head of Product
    '4762c48f-0d6f-4b26-a5d8-8d4ea2a61bad', # Chris Cox, CPO
    'a1f18890-8438-2e76-ddae-487b8ab27778', # Cameron Marlow, Manager, Data Science
]), 'job_type'] = 'employee'

# New start or end date
# ---------------- LinkedIn ----------------
jobs.loc[jobs['job_uuid'] == '5cf7c02c-2b7f-5ffc-3b0a-8e75d1b786c5', 'started_on'] = '2007-09-01' # Michael Gamson

jobs.loc[jobs['job_uuid'] == 'e89399df-33a6-0185-764b-24a3651ba5a9', 'ended_on'] = '2015-10-01' # Konstantin Guericke, Earlybird

jobs.loc[jobs['job_uuid'] == 'e603602f-05c6-34cc-71cd-5b9f9b2ee11a', 'started_on'] = '2003-01-01' # Jean-Luc Vaillant, LinkedIn
jobs.loc[jobs['job_uuid'] == 'e603602f-05c6-34cc-71cd-5b9f9b2ee11a', 'ended_on'] = '2011-05-01' # Jean-Luc Vaillant, LinkedIn

# ---------------- Skype ----------------
jobs.loc[jobs['job_uuid'] == '147cc4cd-d6fd-51e2-dae9-53c170484f24', 'started_on'] = '2006-10-01' # Niklas Zennström, Joost
jobs.loc[jobs['job_uuid'] == '147cc4cd-d6fd-51e2-dae9-53c170484f24', 'ended_on'] = '2009-11-01' # Niklas Zennström, Joost
jobs.loc[jobs['job_uuid'] == '35541d86-11bc-2351-7f6e-19a0b39f42fd', 'started_on'] = '2009-07-08' # Niklas Zennström, Jolicloud
jobs.loc[jobs['job_uuid'] == '7d655b5b-f38c-b605-771d-6d22b41e7aae', 'started_on'] = '2011-03-10' # Niklas Zennström, Rovio
jobs.loc[jobs['job_uuid'] == '17c38eac-8b9a-5171-fb24-ca9292ff5272', 'started_on'] = '2011-11-14' # Niklas Zennström, Wrapp
jobs.loc[jobs['job_uuid'] == 'ddf332d4-f821-95c0-8073-f68eff95d729', 'started_on'] = '2022-01-25' # Niklas Zennström, Orbital Systems
jobs.loc[jobs['job_uuid'] == '0ba6cc15-ba77-4610-b5e6-f4c362728e7e', 'started_on'] = '2018-07-02' # Niklas Zennström, Oden Technologies

jobs.loc[jobs['job_uuid'] == '872491e0-e04e-c98a-aa09-3b69bc1d8bc3', 'started_on'] = '2002-07-01' # Janus Friis, Skype
jobs.loc[jobs['job_uuid'] == '872491e0-e04e-c98a-aa09-3b69bc1d8bc3', 'ended_on'] = '2008-01-01' # Janus Friis, Skype
jobs.loc[jobs['job_uuid'] == '8fecfece-2e7e-9336-285d-0b11b35be9bb', 'started_on'] = '2006-10-01' # Janus Friis, Joost
jobs.loc[jobs['job_uuid'] == '8fecfece-2e7e-9336-285d-0b11b35be9bb', 'ended_on'] = '2009-11-01' # Janus Friis, Joost
jobs.loc[jobs['job_uuid'] == '07ea22ad-467a-7b87-6e9e-7f9ecb50f99a', 'started_on'] = '2007-10-01' # Janus Friis, Atomico
jobs.loc[jobs['job_uuid'] == '07ea22ad-467a-7b87-6e9e-7f9ecb50f99a', 'ended_on'] = '2010-05-01' # Janus Friis, Atomico
jobs.loc[jobs['job_uuid'] == '8b5a4084-6b64-4a2d-0fe0-59e65455b7a4', 'started_on'] = '2014-07-01' # Janus Friis, Starship Tehcnologies
jobs.loc[jobs['job_uuid'] == '8b5a4084-6b64-4a2d-0fe0-59e65455b7a4', 'ended_on'] = 'NAT' # Janus Friis, Starship Tehcnologies

jobs.loc[jobs['job_uuid'] == '79c714d2-eb43-cd80-3c4d-b83af0eb38ed', 'started_on'] = '2016-01-01' # Ahti Heinla, Karma Ventures

jobs.loc[jobs['job_uuid'] == 'a6e3544e-6684-e82d-5767-15e3a5e78bfe', 'started_on'] = '2002-06-01' # Toivo Annus, Skype
jobs.loc[jobs['job_uuid'] == 'a6e3544e-6684-e82d-5767-15e3a5e78bfe', 'ended_on'] = '2005-06-01' # Toivo Annus, Skype
jobs.loc[jobs['job_uuid'] == '2c386735-9a78-0deb-a0d6-c3f9d78c81e7', 'started_on'] = '2007-11-01' # Toivo Annus, Fraktal
jobs.loc[jobs['job_uuid'] == '8ff9b9eb-b3a9-27a7-4fa7-156d299400fe', 'started_on'] = '2007-11-01' # Toivo Annus, Inkspin1
jobs.loc[jobs['job_uuid'] == '15fb1332-1cbb-6d2c-7380-39e19a989d11', 'started_on'] = '2005-11-01' # Toivo Annus, Ambient Sound Investments

jobs.loc[jobs['job_uuid'] == '5b1fdac7-c10d-37e5-425e-efe8b134e210', 'started_on'] = '2016-07-25' # Eileen Burbidge, Tide
jobs.loc[jobs['job_uuid'] == '75eaf3f2-9598-40d2-8151-95dac357eadf', 'started_on'] = '2018-07-07' # Eileen Burbidge, Marshmallow

jobs.loc[jobs['job_uuid'] == 'ee6e231d-dd41-1117-2a26-ec787d5ba005', 'started_on'] = '2002-06-01' # Jaan Tallinn, Skype
jobs.loc[jobs['job_uuid'] == '505440eb-f542-4233-ace1-d2ef25f70b6a', 'started_on'] = '2011-01-01' # Jaan Tallinn, Metaplanet
jobs.loc[jobs['job_uuid'] == '7fb04c7e-265a-4240-a1b5-748bc578a4f6', 'started_on'] = '2014-01-01' # Jaan Tallinn, Future of life institute
jobs.loc[jobs['job_uuid'] == '42903be3-d2ee-4a38-b315-9698a0ea6ef5', 'started_on'] = '2005-11-01' # Jaan Tallinn, Ambient Sound Investments
jobs.loc[jobs['job_uuid'] == '19520fe7-859c-ae53-a5ba-104dbeb69fa9', 'started_on'] = '2016-01-01' # Jaan Tallinn, Karma Ventures
jobs.loc[jobs['job_uuid'] == 'cd9900d6-6655-393b-aff0-7ae46acc409d', 'started_on'] = '2013-07-05' # Jaan Tallinn, Lingvist

# ---------------- eBay ----------------
jobs.loc[jobs['job_uuid'] == 'd938ccec-89ff-5e2e-1788-f306942de482', 'started_on'] = '2007-01-01' # Alexander Samwer, Rocket Internet
jobs.loc[jobs['job_uuid'] == '57fa88d6-d43f-4eb7-ae25-70fdad9d8f24', 'started_on'] = '2016-01-01' # Alexander Samwer, Pacifico Energy Partners
jobs.loc[jobs['job_uuid'] == '57fa88d6-d43f-4eb7-ae25-70fdad9d8f24', 'started_on'] = '2017-01-01' # Alexander Samwer, Pelion Green Future

jobs.loc[jobs['job_uuid'] == '15c57aa4-ff45-73c6-1c52-bf418f89b909', 'ended_on'] = '2000-08-01' # Oliver Samwer, eBay
jobs.loc[jobs['job_uuid'] == '15c57aa4-ff45-73c6-1c52-bf418f89b909', 'job_title'] = 'Managing Director DACH' # Oliver Samwer, eBay

jobs.loc[jobs['job_uuid'] == 'c3566bc5-1a72-db8a-eb11-7b7bbf721c22', 'started_on'] = '2016-01-01' # Marc Samwer, Global Founders Capital

# ---------------- Spotify ----------------
jobs.loc[jobs['job_uuid'] == '519417f6-6512-40ed-8a4b-e6b40abe50e7', 'started_on'] = '2006-01-01' # Martin Lorentzon, Spotify
jobs.loc[jobs['job_uuid'] == '519417f6-6512-40ed-8a4b-e6b40abe50e7', 'ended_on'] = '2005-16-01' # Martin Lorentzon, Spotify
jobs.loc[jobs['job_uuid'] == '519417f6-6512-40ed-8a4b-e6b40abe50e7', 'job_title'] = 'Founder, Director, and Chairman' # Martin Lorentzon, Spotify

jobs.loc[jobs['job_uuid'] == 'eb7604da-77b2-448f-9113-a483e62eb49e', 'started_on'] = '2021-11-01' # Daniel Ek, Helsing

jobs.loc[jobs['job_uuid'] == '9efd68ab-7ff2-4ac0-900b-b23784962e5b', 'job_title'] = 'Vice President of Engineering' # Andreas Blixt, Framer

jobs.loc[jobs['job_uuid'] == 'ef8cb16f-c436-4685-9e3e-0c97540a7cb6', 'started_on'] = '2021-11-01' # Henrik Landgren, Gilion

jobs.loc[jobs['job_uuid'] == '61476d5c-93a2-5111-26b6-934474ad34a0', 'started_on'] = '2007-12-01' # Jonathan Forster, Spotify


# ---------------- iZettle ----------------
jobs.loc[jobs['job_uuid'] == '30af4a3a-dbc7-c2b3-e0dd-e281ed3e0142', 'started_on'] = '2010-02-01' # Magnus Nilsson, iZettle
jobs.loc[jobs['job_uuid'] == '30af4a3a-dbc7-c2b3-e0dd-e281ed3e0142', 'ended_on'] = '2022-04-01' # Magnus Nilsson, iZettle

jobs.loc[jobs['job_uuid'] == '8c835363-77d4-9dca-f173-8fcb698bf02e', 'started_on'] = '2011-03-01' # Johan Bendz, iZettle

jobs.loc[jobs['job_uuid'] == 'b68dfcdf-52fe-4434-8caf-c3fcbcc050a0', 'started_on'] = '2013-12-01' # Johan Bendz, iZettle

jobs.loc[jobs['job_uuid'] == '24bc91df-a45f-438f-93c9-8deafa95acd9', 'started_on'] = '2011-09-01' # Adam von Corswant, iZettle
jobs.loc[jobs['job_uuid'] == '24bc91df-a45f-438f-93c9-8deafa95acd9', 'ended_on'] = '2022-03-01' # Adam von Corswant, iZettle

jobs.loc[jobs['job_uuid'] == '4cdc24ed-aad5-e065-6969-c7005846d0b4', 'started_on'] = '2013-04-01' # Leo Nilsson, iZettle


# ---------------- Kry ----------------
jobs.loc[jobs['job_uuid'] == 'c2ee74d9-ccce-8b14-5626-2a35956d2127', 'job_title'] = 'Co-founder & CTO' # Joachim Hedenius, Kry
jobs.loc[jobs['job_uuid'] == '24bc91df-a45f-438f-93c9-8deafa95acd9', 'ended_on'] = '2020-02-01' # Joachim Hedenius, Kry

jobs.loc[jobs['job_uuid'] == 'fd417a6f-0d5b-4b96-834b-df15513ef40b', 'started_on'] = '2019-09-01' # Josefin Landgård, Mantle

jobs.loc[jobs['job_uuid'] == '50a83f68-5848-42f8-97b7-cba28eed400b', 'started_on'] = '2017-02-01' # Sabina Wizander, Kry
jobs.loc[jobs['job_uuid'] == '50a83f68-5848-42f8-97b7-cba28eed400b', 'ended_on'] = '2021-06-01' # Sabina Wizander, Kry
jobs.loc[jobs['job_uuid'] == '9b0369d2-0bb0-469c-a3dc-325dbce5f0fc', 'started_on'] = '2021-06-01' # Sabina Wizander, Creandum


# ---------------- Netscape ----------------
jobs.loc[jobs['job_uuid'] == '9cff9f59-02ca-580d-2df1-d2238a87115c', 'job_title'] = 'Co-founder & CTO' # Marc Andreessen, Netscape

jobs.loc[jobs['job_uuid'] == '38e1c185-ca67-4f74-9934-8eee7aec3451', 'started_on'] = '2020-12-01' # Omid Kordestani


# ---------------- Meta ----------------
jobs.loc[jobs['job_uuid'] == 'c91252d1-7ca3-a330-00fa-4247b94d4305', 'started_on'] = '2006-03-01' # Jeff Hammerbacher
jobs.loc[jobs['job_uuid'] == 'c91252d1-7ca3-a330-00fa-4247b94d4305', 'ended_on'] = '2008-09-01' # Jeff Hammerbacher



new_records = pd.DataFrame([
    # ---------------- PayPal ----------------
    # Jason Portnoy
    # Oakhouse Partners
    {
        'job_uuid': '0191215b-44ca-72a2-99f4-ab6ea063c643',
        'person_uuid': '0191214a-e133-78ba-84ce-3777ffbd4b57',
        'org_uuid': '5c26c58f-6d80-43e8-8022-24c2d769af0a',
        'started_on': '2012-01-01',
        'ended_on': 'NAT',
        'is_current': True,
        'job_title': 'Founder and Managing Partner',
        'job_type': 'executive'
    },
    # Practice Fusion
    {
        'job_uuid': '0191215f-a95c-7d01-9594-0e7437952450',
        'person_uuid': '0191214a-e133-78ba-84ce-3777ffbd4b57',
        'org_uuid': '33cb1e8f-e5c4-1413-c6ad-e3a70219d958',
        'started_on': '2010-01-11',
        'ended_on': '2011-01-11',
        'is_current': False,
        'job_title': 'Chief Financial Officer',
        'job_type': 'executive'
    },
    # Palantir
    {
        'job_uuid': '01912187-08e0-7e43-a035-53894eb7ae13',
        'person_uuid': '0191214a-e133-78ba-84ce-3777ffbd4b57',
        'org_uuid': '4b9df299-4ef2-1fad-607e-278d90eb69ab',
        'started_on': '2007-03-01',
        'ended_on': '2010-01-01',
        'is_current': False,
        'job_title': 'Chief Financial Officer',
        'job_type': 'executive'
    },
    # Clarium Capital
    {
        'job_uuid': '01912187-2807-796c-a040-9f7b5631aa14',
        'person_uuid': '0191214a-e133-78ba-84ce-3777ffbd4b57',
        'org_uuid': '7bcc0a57-a5f6-7ef5-0945-28bcbfc54559',
        'started_on': '2003-01-01',
        'ended_on': '2007-03-01',
        'is_current': False,
        'job_title': 'CFO/SVP Finance',
        'job_type': 'executive'
    },
    # PayPal
    {
        'job_uuid': '01912187-3919-79e1-b61e-22f10f06b7be',
        'person_uuid': '0191214a-e133-78ba-84ce-3777ffbd4b57',
        'org_uuid': '96ab87ca-00b5-2ebc-f218-86262954e320',
        'started_on': '2000-01-01',
        'ended_on': '2003-01-01',
        'is_current': False,
        'job_title': 'VP of Financial Planning and Analysis',
        'job_type': 'executive'
    },
    # John Muller
    # Stripe
    {
        'job_uuid': '01912187-5035-7d3b-a79f-f16de767c20a',
        'person_uuid': '980efaac-fba9-6eb7-cc32-52891edf10df',
        'org_uuid': '6f83ddd7-d637-61f8-06b2-438a0037605f',
        'started_on': '2022-03-01',
        'ended_on': 'NAT',
        'is_current': True,
        'job_title': 'Lead Product Counsel',
        'job_type': 'executive'
    },
    # ---------------- LinkedIn ----------------
    # Jay Kreps
    {
        'job_uuid': '01912dc2-3876-7cd6-bbdc-0f2592116931',
        'person_uuid': 'd3c58ad1-6a90-712c-c5e7-1dbb79c3f324',
        'org_uuid': '86da6213-5b43-6419-4047-472102ccf66f',
        'started_on': '2007-06-01',
        'ended_on': '2014-09-01',
        'is_current': False,
        'job_title': 'Principal Staff Engineer',
        'job_type': 'executive'
    },
    # ---------------- Spotify ----------------
    # Sophia Bendz
    # Spotify
    {
        'job_uuid': '0191834a-97f9-7987-b916-4a9926fd5644',
        'person_uuid': '26d49c52-f508-c857-5bfc-2e3ef57b17cf',
        'org_uuid': '022417b5-4980-6c54-0f3c-6736bbbb1a5e',
        'started_on': '2007-05-01',
        'ended_on': '2014-12-01',
        'is_current': False,
        'job_title': 'Global Marketing Director',
        'job_type': 'executive'
    },
    # Avanza
    {
        'job_uuid': '01918350-a7d1-7a6b-a591-108bdd2ae1a1',
        'person_uuid': '26d49c52-f508-c857-5bfc-2e3ef57b17cf',
        'org_uuid': 'ad648ff9-4a94-3d4d-9d0d-41324dec108b',
        'started_on': '2015-12-01',
        'ended_on': '2019-03-01',
        'is_current': False,
        'job_title': 'Member of the Board of Directors',
        'job_type': 'board_member'
    },
    # Telia
    {
        'job_uuid': '01918353-41e6-7232-834c-f1b32359537d',
        'person_uuid': '26d49c52-f508-c857-5bfc-2e3ef57b17cf',
        'org_uuid': '0f22c738-8420-e930-76b2-c42b3d8d09ca',
        'started_on': '2017-01-01',
        'ended_on': '2019-01-01',
        'is_current': False,
        'job_title': 'Member of the Advisory Board',
        'job_type': 'board_member'
    },
    # Kindred Group
    {
        'job_uuid': '01918354-bdb2-78de-ba85-47f018182be3',
        'person_uuid': '26d49c52-f508-c857-5bfc-2e3ef57b17cf',
        'org_uuid': 'c018d622-e38e-eaea-6f86-6e7b46840bf1',
        'started_on': '2014-05-01',
        'ended_on': '2018-05-01',
        'is_current': False,
        'job_title': 'Member of the Board of Directors',
        'job_type': 'board_member'
    },
    # Adam Williams
    # Takumi
    {
        'job_uuid': '01918388-581b-7678-b74a-156bb93e0f81',
        'person_uuid': 'ae779864-fdbc-c059-d4df-4a59e9531c9c',
        'org_uuid': '75bf879d-5951-8ffe-25b2-17fbb14f572c',
        'started_on': '2018-01-01',
        'ended_on': '2020-02-01',
        'is_current': False,
        'job_title': 'CEO',
        'job_type': 'executive'
    },
    # Inflencer Intelligence
    {
        'job_uuid': '019183ba-d2e0-70a1-957c-3e5f11fbbc48',
        'person_uuid': 'ae779864-fdbc-c059-d4df-4a59e9531c9c',
        'org_uuid': '815aac34-212d-4278-9fd9-1ffd5e00d4d8',
        'started_on': '2022-01-01',
        'ended_on': '2023-02-01',
        'is_current': False,
        'job_title': 'Managing Director',
        'job_type': 'executive'
    },
    # Outra
    {
        'job_uuid': '019183bb-e399-76da-8e2c-f70fbd85ec55',
        'person_uuid': 'ae779864-fdbc-c059-d4df-4a59e9531c9c',
        'org_uuid': '727128af-4edb-49ea-bfe9-c7ac92a8e4ed',
        'started_on': '2023-01-01',
        'ended_on': '2023-10-01',
        'is_current': False,
        'job_title': 'Chief Commercial Officer',
        'job_type': 'executive'
    },
    # Upscalers
    {
        'job_uuid': '019183bd-63ac-7837-bdb7-e22de877af5b',
        'person_uuid': 'ae779864-fdbc-c059-d4df-4a59e9531c9c',
        'org_uuid': '1fed2b6a-9101-48ff-bab3-e0fdf26266f2',
        'started_on': '2021-09-01',
        'ended_on': 'NAT',
        'is_current': True,
        'job_title': 'Investor and Community Member',
        'job_type': 'executive'
    },
    # Oskar Serrander
    # Spotify
    {
        'job_uuid': '019183ce-a825-7cb1-bf5c-1a5ee4d2e36e',
        'person_uuid': '8246831f-24b8-8645-27e1-f0672bf062d4',
        'org_uuid': '022417b5-4980-6c54-0f3c-6736bbbb1a5e',
        'started_on': '2009-01-01',
        'ended_on': '2014-01-01',
        'is_current': False,
        'job_title': 'Director of Sales & Marketing Partnerships',
        'job_type': 'executive'
    },
    # Quixey
    {
        'job_uuid': '019183d1-e785-78e4-ab96-383a8c339853',
        'person_uuid': '8246831f-24b8-8645-27e1-f0672bf062d4',
        'org_uuid': 'b66114e0-49ca-ca9a-a337-f1d637e81a00',
        'started_on': '2014-01-01',
        'ended_on': '2015-01-01',
        'is_current': False,
        'job_title': 'Director, Monetization / Interim CRO',
        'job_type': 'executive'
    },
    # iHeartMedia
    {
        'job_uuid': '019183d3-8f4e-760b-9074-5a3819e8e005',
        'person_uuid': '8246831f-24b8-8645-27e1-f0672bf062d4',
        'org_uuid': 'b5c69db1-5367-8301-1945-3704bdf67bf6',
        'started_on': '2015-01-01',
        'ended_on': '2017-01-01',
        'is_current': False,
        'job_title': 'VP Business Development',
        'job_type': 'executive'
    },
    # Jonathan Forster
    # Kry
    {
        'job_uuid': '0191848d-3be5-7de0-9840-14f8bc5540e3',
        'person_uuid': '3a8d3f51-6720-65da-65a1-170eefc68d21',
        'org_uuid': '7b74b50c-4468-b7ec-2ff9-bf3286a399c9',
        'started_on': '2017-09-01',
        'ended_on': '2018-02-01',
        'is_current': False,
        'job_title': 'VP Marketing (Interim)',
        'job_type': 'executive'
    },
    # Marshall
    {
        'job_uuid': '0191848f-7d93-7a17-8b43-3b490930841c',
        'person_uuid': '3a8d3f51-6720-65da-65a1-170eefc68d21',
        'org_uuid': 'fd459c19-acff-491b-8b3e-40d0be22fbce',
        'started_on': '2019-08-01',
        'ended_on': 'NAT',
        'is_current': True,
        'job_title': 'Chairman of the Board of Directors',
        'job_type': 'board_member'
    },
    # ---------------- iZettle ----------------
    # Greater Than
    {
        'job_uuid': '019183f9-e6b7-71fa-9343-360bfcb12ab1',
        'person_uuid': '5e6974af-5fcd-ff4e-b921-09124dda0bc4',
        'org_uuid': 'edbc4643-999a-832c-9d9f-cb464ea04ca5',
        'started_on': '2021-11-01',
        'ended_on': '2023-05-01',
        'is_current': False,
        'job_title': 'Board Member',
        'job_type': 'board_member'
    },
    # Ancon
    {
        'job_uuid': '019183fb-eab0-78dd-8337-e330109ad776',
        'person_uuid': '5e6974af-5fcd-ff4e-b921-09124dda0bc4',
        'org_uuid': '7e80515b-5eb6-4bf0-878f-97051cd4313f',
        'started_on': '2023-01-01',
        'ended_on': 'NAT',
        'is_current': True,
        'job_title': 'Board Member',
        'job_type': 'board_member'
    },
    # Peder Stahle
    # iZettle
    {
        'job_uuid': '019184b7-7b44-7ddc-9a18-cc5d3c8bc5da',
        'person_uuid': '763c8fbd-4285-4209-a9bb-9da60b3bd777',
        'org_uuid': '6093de34-5382-f34c-a207-efa92470048b',
        'started_on': '2010-04-01',
        'ended_on': '2016-12-01',
        'is_current': False,
        'job_title': 'Chief Product Officer',
        'job_type': 'executive'
    },
    # Kry
    {
        'job_uuid': '019184b7-8247-7996-a277-8fa1a1afd5d6',
        'person_uuid': '763c8fbd-4285-4209-a9bb-9da60b3bd777',
        'org_uuid': '7b74b50c-4468-b7ec-2ff9-bf3286a399c9',
        'started_on': '2017-02-01',
        'ended_on': '2023-01-01',
        'is_current': False,
        'job_title': 'Chief Product Officer',
        'job_type': 'executive'
    },
    # Adam von Corswant
    # Terra Labs
    {
        'job_uuid': '019184c2-b57e-793c-8708-8ac330dc151c',
        'person_uuid': '06331f9a-81e7-41cf-a8ff-94bf6030e3ff',
        'org_uuid': 'd626d3dd-d5cd-4507-9b29-fd82308b84cd',
        'started_on': '2023-03-01',
        'ended_on': 'NAT',
        'is_current': True,
        'job_title': 'Co-founder and CTO',
        'job_type': 'executive'
    },
    # Ron Stolero
    # iZettle
    {
        'job_uuid': '019184c6-0803-7e88-a0ac-5e31d46bb843',
        'person_uuid': '480a5e71-1386-4ee0-8a07-23762a07ff0b',
        'org_uuid': '6093de34-5382-f34c-a207-efa92470048b',
        'started_on': '2013-08-01',
        'ended_on': '2021-04-01',
        'is_current': False,
        'job_title': 'Data Lead',
        'job_type': 'executive'
    },
    # Fever
    {
        'job_uuid': '019184c7-828e-75dd-b9ed-44813e91f058',
        'person_uuid': '480a5e71-1386-4ee0-8a07-23762a07ff0b',
        'org_uuid': '2c4bdca3-1244-4d89-8912-931db7573343',
        'started_on': '2022-01-01',
        'ended_on': 'NAT',
        'is_current': True,
        'job_title': 'Co-founder',
        'job_type': 'executive'
    },
    # Klas Johansson
    # Fever
    {
        'job_uuid': '019184d0-afae-79ac-b577-d832b024f0a2',
        'person_uuid': '019184cd-cee7-73cd-ae18-a4a414f4035f',
        'org_uuid': '2c4bdca3-1244-4d89-8912-931db7573343',
        'started_on': '2022-01-01',
        'ended_on': 'NAT',
        'is_current': True,
        'job_title': 'Co-founder & CEO',
        'job_type': 'executive'
    },
    # Ruben Flam
    # Fever
    {
        'job_uuid': '019184d0-c704-7a99-a0b6-b402b77dbef5',
        'person_uuid': '019184cf-f87f-7c70-b207-004b1359a76f',
        'org_uuid': '2c4bdca3-1244-4d89-8912-931db7573343',
        'started_on': '2022-01-01',
        'ended_on': 'NAT',
        'is_current': True,
        'job_title': 'Co-founder',
        'job_type': 'executive'
    },
    # ---------------- Kry ----------------
    # Fredrik Jung Abbou
    # Norrsken Launcher
    {
        'job_uuid': '01918590-1839-7004-8365-771a950a4459',
        'person_uuid': '5c0a828d-ea80-44a7-823f-1389786c38ef',
        'org_uuid': '01053f4e-712c-4cce-a2aa-7654c0398f6c',
        'started_on': '2023-01-01',
        'ended_on': 'NAT',
        'is_current': True,
        'job_title': 'Co-founder',
        'job_type': 'executive'
    },
    # Sabina Wizander
    # Norrsken Impact Accelerator
    {
        'job_uuid': '0191882b-3b11-7fc5-8732-d5443dcf4aba',
        'person_uuid': '0eb55aec-3c56-0415-4496-09b5b5cf16f0',
        'org_uuid': 'e0891a23-4bb6-4028-b93a-ad8fc319ba4c',
        'started_on': '2021-01-01',
        'ended_on': '2022-01-01',
        'is_current': False,
        'job_title': 'Co-founder Norrsken Impact Accelerator',
        'job_type': 'executive'
    },
    # Passionfroot
    {
        'job_uuid': '0191882e-fcc7-70db-ae7a-6f9f177c9316',
        'person_uuid': '0eb55aec-3c56-0415-4496-09b5b5cf16f0',
        'org_uuid': 'aa59ea70-9974-4fcc-859f-d9fe11acc92b',
        'started_on': '2022-01-01',
        'ended_on': 'NAT',
        'is_current': True,
        'job_title': 'Investor & Board Member',
        'job_type': 'board_member'
    },
    # Twirl
    {
        'job_uuid': '0191882f-798f-7907-a843-9064dfb539a7',
        'person_uuid': '0eb55aec-3c56-0415-4496-09b5b5cf16f0',
        'org_uuid': '8af25ece-f9b2-4389-a671-e5fe00839d5c',
        'started_on': '2022-06-01',
        'ended_on': 'NAT',
        'is_current': True,
        'job_title': 'Investor & Board Member',
        'job_type': 'board_member'
    },
    # Enode
    {
        'job_uuid': '0191882f-9c1d-7c88-b4fb-3c0111dff5ee',
        'person_uuid': '0eb55aec-3c56-0415-4496-09b5b5cf16f0',
        'org_uuid': '0f801da2-be3d-4973-b5f1-b96b374b9159',
        'started_on': '2022-08-01',
        'ended_on': 'NAT',
        'is_current': True,
        'job_title': 'Investor & Board Member',
        'job_type': 'board_member'
    },
    # Prewave
    {
        'job_uuid': '0191882f-a7ce-7beb-af2f-1f79b9b7293c',
        'person_uuid': '0eb55aec-3c56-0415-4496-09b5b5cf16f0',
        'org_uuid': '760f0aa3-c4e3-4e2b-b79e-bd951e99cb0c',
        'started_on': '2023-05-01',
        'ended_on': 'NAT',
        'is_current': True,
        'job_title': 'Investor & Board Member',
        'job_type': 'board_member'
    },
    # ---------------- Netscape ----------------
    # Omid Kordestani
    # Pearson
    {
        'job_uuid': '019252af-48c4-7169-8104-795ce84d053b',
        'person_uuid': 'b42cd994-c370-73d6-1702-1f590da4f71b',
        'org_uuid': '5791d792-970c-8055-0680-20e417976ed7',
        'started_on': '2022-05-01',
        'ended_on': 'NAT',
        'is_current': True,
        'job_title': 'Chairman',
        'job_type': 'board_member'
    },
    # Eckart Walther
    # Uber
    {
        'job_uuid': '01925311-20c8-7a8f-90d9-ee9613cd9c5a',
        'person_uuid': 'c1c05db8-0b79-bcf6-45c1-1e7b8ed3c6b8',
        'org_uuid': '1eb37109-3b93-01a9-177f-fee2cb1bfcdc',
        'started_on': '2018-01-01',
        'ended_on': '2020-01-01',
        'is_current': False,
        'job_title': 'Head of Product Management',
        'job_type': 'executive'
    },
    # ---------------- Klarna ----------------
    # Mikael Hussain
    # Klarna
    {
        'job_uuid': '0192c810-babc-73b8-b90e-11a227c9fbba',
        'person_uuid': '73e47535-737d-40dd-8d6e-5185a4d5d77d',
        'org_uuid': '2cc3a5de-2303-aa00-cd1a-50bd96420392',
        'started_on': '2009-12-01',
        'ended_on': '2017-03-01',
        'is_current': False,
        'job_title': 'Vice President Credit',
        'job_type': 'executive'
    },
    # Sven Perkmann
    # Klarna
    {
        'job_uuid': '0192c814-4e5b-745f-a366-ae17305d5485',
        'person_uuid': 'df3fcb4a-a555-4a3b-8832-74f57933710a',
        'org_uuid': '2cc3a5de-2303-aa00-cd1a-50bd96420392',
        'started_on': '2010-01-01',
        'ended_on': '2016-12-01',
        'is_current': False,
        'job_title': 'Head of Risk Product',
        'job_type': 'executive'
    },
])



jobs = pd.concat([jobs, new_records], ignore_index = True)

jobs['started_on'] = pd.to_datetime(jobs['started_on'], errors = 'coerce')
jobs['ended_on'] = pd.to_datetime(jobs['ended_on'], errors = 'coerce')

jobs['job_title'] = jobs['job_title'].astype(str)

new_records

,job_uuid,person_uuid,org_uuid,started_on,ended_on,is_current,job_title,job_type
0,0191215b-44ca-72a2-99f4-ab6ea063c643,0191214a-e133-78ba-84ce-3777ffbd4b57,5c26c58f-6d80-43e8-8022-24c2d769af0a,2012-01-01,NAT,True,Founder and Managing Partner,executive
1,0191215f-a95c-7d01-9594-0e7437952450,0191214a-e133-78ba-84ce-3777ffbd4b57,33cb1e8f-e5c4-1413-c6ad-e3a70219d958,2010-01-11,2011-01-11,False,Chief Financial Officer,executive
2,01912187-08e0-7e43-a035-53894eb7ae13,0191214a-e133-78ba-84ce-3777ffbd4b57,4b9df299-4ef2-1fad-607e-278d90eb69ab,2007-03-01,2010-01-01,False,Chief Financial Officer,executive
3,01912187-2807-796c-a040-9f7b5631aa14,0191214a-e133-78ba-84ce-3777ffbd4b57,7bcc0a57-a5f6-7ef5-0945-28bcbfc54559,2003-01-01,2007-03-01,False,CFO/SVP Finance,executive
4,01912187-3919-79e1-b61e-22f10f06b7be,0191214a-e133-78ba-84ce-3777ffbd4b57,96ab87ca-00b5-2ebc-f218-86262954e320,2000-01-01,2003-01-01,False,VP of Financial Planning and Analysis,executive
5,01912187-5035-7d3b-a79f-f16de767c20a,980efaac-fba9-6eb7-cc32-52891edf10df,6f83ddd7-d637-61f8-06b2-438a0037605f,2022-03-01,NAT,True,Lead Product Counsel,executive
6,01912dc2-3876-7cd6-bbdc-0f2592116931,d3c58ad1-6a90-712c-c5e7-1dbb79c3f324,86da6213-5b43-6419-4047-472102ccf66f,2007-06-01,2014-09-01,False,Principal Staff Engineer,executive
7,0191834a-97f9-7987-b916-4a9926fd5644,26d49c52-f508-c857-5bfc-2e3ef57b17cf,022417b5-4980-6c54-0f3c-6736bbbb1a5e,2007-05-01,2014-12-01,False,Global Marketing Director,executive
8,01918350-a7d1-7a6b-a591-108bdd2ae1a1,26d49c52-f508-c857-5bfc-2e3ef57b17cf,ad648ff9-4a94-3d4d-9d0d-41324dec108b,2015-12-01,2019-03-01,False,Member of the Board of Directors,board_member
9,01918353-41e6-7232-834c-f1b32359537d,26d49c52-f508-c857-5bfc-2e3ef57b17cf,0f22c738-8420-e930-76b2-c42b3d8d09ca,2017-01-01,2019-01-01,False,Member of the Advisory Board,board_member


## Funding Rounds

In [11]:
funding_rounds = pd.read_csv(data_path + '/funding_rounds.csv')[['uuid', 'investment_type', 'announced_on', 'raised_amount_usd', 'org_uuid']]

funding_rounds = funding_rounds.rename(columns = {'uuid': 'funding_round_uuid'})

funding_rounds = funding_rounds.assign(announced_on = pd.to_datetime(funding_rounds['announced_on'], errors = 'coerce'))

funding_rounds.head()

,funding_round_uuid,investment_type,announced_on,raised_amount_usd,org_uuid
0,8a945939-18e0-cc9d-27b9-bf33817b2818,angel,2004-09-01,500000.0,df662812-7f97-0b43-9d3e-12f64f504fbb
1,d950d7a5-79ff-fb93-ca87-13386b0e2feb,series_a,2005-05-01,12700000.0,df662812-7f97-0b43-9d3e-12f64f504fbb
2,6fae3958-a001-27c0-fb7e-666266aedd78,series_b,2006-04-01,27500000.0,df662812-7f97-0b43-9d3e-12f64f504fbb
3,bcd5a63d-ed99-6963-0dd2-e36f6582f846,series_b,2006-05-01,10500000.0,f53cb4de-236e-0b1b-dee8-7104a8b018f9
4,60e6afd9-1215-465a-dd17-0ed600d4e29b,series_a,2007-01-17,NaN,4111dc8b-c0df-2d24-ed33-30cd137b3098


## Investments

In [12]:
investments = pd.read_csv(data_path + '/investments.csv')[['uuid', 'funding_round_uuid', 'investor_uuid']]

investments = investments.rename(columns = {'uuid': 'investment_uuid'})

investments.head()

,investment_uuid,funding_round_uuid,investor_uuid
0,524986f0-3049-54a4-fa72-f60897a5e61d,d950d7a5-79ff-fb93-ca87-13386b0e2feb,b08efc27-da40-505a-6f9d-c9e14247bf36
1,6556ab92-6465-25aa-1ffc-7f8b4b09a476,6fae3958-a001-27c0-fb7e-666266aedd78,e2006571-6b7a-e477-002a-f7014f48a7e3
2,0216e06a-61f8-9cf1-19ba-20811229c53e,6fae3958-a001-27c0-fb7e-666266aedd78,8d5c7e48-82da-3025-dd46-346a31bab86f
3,dadd7d86-520d-5e35-3033-fc1d8792ab91,bcd5a63d-ed99-6963-0dd2-e36f6582f846,7ca12f7a-2f8e-48b4-a8d1-1a33a0e275b9
4,581c4b38-9653-7117-9bd4-7ffe5c7eba69,60e6afd9-1215-465a-dd17-0ed600d4e29b,fb2f8884-ec07-895a-48d7-d9a9d4d7175c


In [13]:
investment_partners = pd.read_csv(data_path + '/investment_partners.csv')[['uuid', 'funding_round_uuid', 'partner_uuid']]

investment_partners = investment_partners.rename(columns = {'uuid': 'investment_partner_uuid'})

investment_partners.head()

,investment_partner_uuid,funding_round_uuid,partner_uuid
0,524986f0-3049-54a4-fa72-f60897a5e61d,d950d7a5-79ff-fb93-ca87-13386b0e2feb,2d78d1e7-203c-3eb6-bf1b-c51f10e0679b
1,524986f0-3049-54a4-fa72-f60897a5e61d,d950d7a5-79ff-fb93-ca87-13386b0e2feb,eaf6c243-d355-32f3-e23a-2a5fc82e8b34
2,6556ab92-6465-25aa-1ffc-7f8b4b09a476,6fae3958-a001-27c0-fb7e-666266aedd78,478e7efd-bec4-b9f5-304b-cffedc1fc012
3,0216e06a-61f8-9cf1-19ba-20811229c53e,6fae3958-a001-27c0-fb7e-666266aedd78,0f9f3c05-cb79-f58f-6cc5-98ddc8382d4f
4,dadd7d86-520d-5e35-3033-fc1d8792ab91,bcd5a63d-ed99-6963-0dd2-e36f6582f846,ea9f4980-600c-84f4-a5d6-b4f8c2f787fb


## Acquisitions

In [14]:
acquisitions = pd.read_csv(data_path + '/acquisitions.csv')[['acquiree_uuid', 'acquirer_uuid', 'type', 'acquirer_name', 'acquired_on', 'acquisition_type', 'price_usd']]

acquisitions = acquisitions.rename(columns = {'acquiree_uuid': 'org_uuid', 'type': 'exit_type', 'acquired_on': 'exit_date', 'price_usd': 'exit_valuation'})

acquisitions = acquisitions.assign(exit_date = pd.to_datetime(acquisitions['exit_date'], errors = 'coerce'))

acquisitions['exit_valuation'] = acquisitions['exit_valuation'].astype('Int64')

acquisitions.head()

,org_uuid,acquirer_uuid,exit_type,acquirer_name,exit_date,acquisition_type,exit_valuation
0,180ebf67-68d0-2316-e93d-8e1e546330ba,d70777cc-14bd-2416-0692-5a483781b78b,acquisition,Fox Interactive Media,2007-05-30,NaN,<NA>
1,5b05e013-a448-3a0b-d872-a6ae668e1192,6acfa7da-1dbd-936e-d985-cf07a1b27711,acquisition,Google,2007-07-01,NaN,60000000
2,8249dffa-1ca6-6f99-9f76-d56c83f85f2d,f09c1228-2e7d-1889-6647-ba5021b2e4ea,acquisition,CBS Entertainment,2007-05-01,NaN,280000000
3,10dd03fa-69ff-3a82-6321-c6b16c9a9f41,6acfa7da-1dbd-936e-d985-cf07a1b27711,acquisition,Google,2007-05-23,acquisition,100000000
4,0af10345-613d-e144-f8bd-b62e288985a0,b5a96cd7-044d-70f0-04c5-f125e57a4b35,acquisition,Scripps Networks,2007-07-01,NaN,<NA>


# IPOs

In [15]:
ipos = pd.read_csv(data_path + '/ipos.csv')[['org_uuid', 'type', 'went_public_on', 'valuation_price_usd']]

ipos = ipos.rename(columns = {'uuid': 'investment_partner_uuid', 'type': 'exit_type', 'went_public_on': 'exit_date', 'valuation_price_usd': 'exit_valuation'})

ipos = ipos.assign(exit_date = pd.to_datetime(ipos['exit_date'], errors = 'coerce'))

ipos['exit_valuation'] = ipos['exit_valuation'].astype('Int64')

ipos.head()

,org_uuid,exit_type,exit_date,exit_valuation
0,fd80725f-53fc-7009-9878-aeecf1e9ffbb,ipo,1986-03-13,<NA>
1,756936c0-c335-f0ae-0a3d-fe26bdff5695,ipo,1978-01-13,<NA>
2,73296f0d-85a5-78d5-90b3-86c5f8981ba9,ipo,2006-10-22,160000000
3,ff8439cf-097c-a88a-9bb9-dd83d23aa14b,ipo,1999-12-02,<NA>
4,ab8e5ba4-df5d-121b-93b6-eae7a0c89245,ipo,1988-08-12,6000000000


In [16]:
acquisitons_ipos = pd.concat([acquisitions, ipos.drop_duplicates(subset = ['org_uuid', 'exit_date'])], ignore_index = True).sort_values(by = 'exit_date', ascending = False).drop_duplicates(subset = ['org_uuid'])

acquisitons_ipos.head()

,org_uuid,acquirer_uuid,exit_type,acquirer_name,exit_date,acquisition_type,exit_valuation
169753,c3e3f36f-8ec3-41d7-8675-089a616d7acb,a1f0ae85-de5a-43a4-9b4c-661811c8cddc,acquisition,Bikaji Foods,2024-08-23,acquisition,7212225
169705,fee4180b-8666-a963-273f-a63bfbdc88a1,29bdc824-a1c6-4284-b84d-27fe02ca1857,acquisition,Husqvarna Group,2024-08-23,acquisition,<NA>
169738,b41c4b6a-24b8-4fc4-b596-6add57442f6e,8b8a52ee-1a15-4f8b-9ebc-6e8970de8ef4,acquisition,Inspired Pet Nutrition,2024-08-23,acquisition,<NA>
169733,a3a102b0-f7c3-46d7-9e08-d1d27482556b,f3a59a4a-c42f-4f23-aac3-9bee81a4fa11,acquisition,IN2 Group,2024-08-22,acquisition,<NA>
169718,770c6dcb-0810-43e1-88e2-5614590a2909,6645bbf2-94e6-2349-4746-ea9d7f61a8a4,acquisition,Morgenthaler Private Equity,2024-08-22,acquisition,<NA>


# Defining Constants

In [17]:
# Years after foundation to consider
years_founding_cutoff = 4

# Define the priority list
relation_priority = ['executive', 'board_member', 'advisor', 'investor']

# Create a mapping for priority
relation_priority_map = {value: index for index, value in enumerate(relation_priority)}

# Creating Main Dataset

## alumni_master - Information about everyone's jobs and subsequent jobs/investments

In [63]:
investments_info = pd.merge(
    investments,
    funding_rounds,
    how = 'left',
    on = 'funding_round_uuid'
).rename(columns = {'investor_uuid': 'person_uuid', 'announced_on': 'started_on'})[['person_uuid', 'org_uuid', 'started_on']]

investments_info = investments_info.assign(job_type = 'investor')
investments_info = investments_info.assign(job_title = 'investor')

investments_info

,person_uuid,org_uuid,started_on,job_type,job_title
0,b08efc27-da40-505a-6f9d-c9e14247bf36,df662812-7f97-0b43-9d3e-12f64f504fbb,2005-05-01,investor,investor
1,e2006571-6b7a-e477-002a-f7014f48a7e3,df662812-7f97-0b43-9d3e-12f64f504fbb,2006-04-01,investor,investor
2,8d5c7e48-82da-3025-dd46-346a31bab86f,df662812-7f97-0b43-9d3e-12f64f504fbb,2006-04-01,investor,investor
3,7ca12f7a-2f8e-48b4-a8d1-1a33a0e275b9,f53cb4de-236e-0b1b-dee8-7104a8b018f9,2006-05-01,investor,investor
4,fb2f8884-ec07-895a-48d7-d9a9d4d7175c,4111dc8b-c0df-2d24-ed33-30cd137b3098,2007-01-17,investor,investor
...,...,...,...,...,...
1090634,73633ee4-ea65-2967-6c5d-9b5fec7d2d5e,d19f6be2-3e55-4d8c-a065-c5d6ea9df120,2023-05-01,investor,investor
1090635,d702bd53-d9be-4349-b98e-930ccea97416,ba517f36-6db8-4d2e-a1f4-a7cf76f8c7b5,2024-04-15,investor,investor
1090636,436b43be-3e9b-4c61-8bed-9d6301fc1c5e,1180244d-4d42-b738-6c84-bfbca0fe58ef,2022-01-10,investor,investor
1090637,2f1d01cd-0471-fdb7-3a4e-4414f9c37c3a,1180244d-4d42-b738-6c84-bfbca0fe58ef,2022-01-10,investor,investor


In [80]:
all_info

,person_uuid,org_uuid,started_on,ended_on,job_title,job_type,org_name,org_country_code,org_city,founded_on,short_description,total_funding_usd,org_logo_url,person_name,person_logo_url,acquirer_uuid,exit_type,acquirer_name,exit_date,acquisition_type,exit_valuation
0,ed13cd36-fe2b-3707-197b-0c2d56e37a71,e1393508-30ea-8a36-3f96-dd3226033abd,2005-10-01,2014-06-01,Co-Founder and CEO,executive,Wetpaint,USA,New York,2005-06-01,Wetpaint offers an online social publishing platform that helps digital publishers grow their customer base.,39750000,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180177/2036b3394a37152e0ff69f27c71bc883.jpg,Ben Elowitz,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180224/56303d2f4b99dd1dcc8abf17ba3fd8bf.jpg,4be40c73-8cd5-fdb5-b1d5-3597a0d92afa,acquisition,"Viggle, Inc.",2013-12-16,acquisition,30000000
1,5ceca97b-493c-1446-6249-5aaa33464763,e1393508-30ea-8a36-3f96-dd3226033abd,NaT,NaT,VP Marketing,executive,Wetpaint,USA,New York,2005-06-01,Wetpaint offers an online social publishing platform that helps digital publishers grow their customer base.,39750000,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180177/2036b3394a37152e0ff69f27c71bc883.jpg,Kevin Flaherty,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180225/a307ba697e0f042623f05f35477bf495.jpg,4be40c73-8cd5-fdb5-b1d5-3597a0d92afa,acquisition,"Viggle, Inc.",2013-12-16,acquisition,30000000
2,6e1bca72-a865-b518-b305-31214ce2d1b0,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,2006-03-01,NaT,VP Business Development,executive,Zoho,USA,Pleasanton,1996-03-17,"Zoho offers a suite of business, collaboration, and productivity applications.",<NA>,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180181/f8aaab73f17af0296eba5deda7a5b95b.png,Ian Wenig,https://images.crunchbase.com/image/upload/t_cb-default-original/v1442309935/yiajbpfbxyopc5l4zs4t.png,NaN,NaN,NaN,NaT,NaN,<NA>
3,c92a1f00-8c19-bf2e-0f28-dbbd383dc968,5f2b40b8-d1b3-d323-d81a-b7a8e89553d0,2005-07-01,2010-04-05,CEO,executive,Digg,USA,New York,2004-10-11,"Digg Inc. operates a website that enables its users to find, read, and share the most interesting and talked about stories on the internet.",49000000,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180182/e77f8f561153ffb45a9ffd538978380d.jpg,Jay Adelson,https://images.crunchbase.com/image/upload/t_cb-default-original/v1430497646/bynhvcvn56oi19ma4mbu.jpg,dbc401a6-2bd6-74b4-07f5-aa54c73a57ab,acquisition,BuySellAds,2018-04-25,acquisition,<NA>
4,80d25c23-9726-9dda-5852-39cdf4810ea5,5f2b40b8-d1b3-d323-d81a-b7a8e89553d0,NaT,NaT,Systems Engineering Manager,executive,Digg,USA,New York,2004-10-11,"Digg Inc. operates a website that enables its users to find, read, and share the most interesting and talked about stories on the internet.",49000000,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180182/e77f8f561153ffb45a9ffd538978380d.jpg,Ron Gorodetzky,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180229/c6450df164e1630d9f91d6f554ffcaf0.jpg,dbc401a6-2bd6-74b4-07f5-aa54c73a57ab,acquisition,BuySellAds,2018-04-25,acquisition,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2426247,4ae6f373-2f90-f21d-5f37-e8d21f1104de,6c63e24c-7198-49ac-ba96-9da0c0697255,2023-01-09,NaT,investor,investor,Driftio,SWE,Stockholm,2023-01-09,The Driftio apps and web service offer easy and secure sharing of personal financial information.,300000,https://images.crunchbase.com/image/upload/t_cb-default-original/gtozfdv8yno1gljck5tr,Peter Montgomery,https://images.crunchbase.com/image/upload/t_cb-default-original/gmmmmnmryoh6ea2nler3,NaN,NaN,NaN,NaT,NaN,<NA>
2426248,cba135a1-5b2d-4a24-b402-7d168e258739,956f26a8-866a-4754-84a3-ef0f6f19e6fa,2023-11-08,NaT,investor,investor,CLIP,USA,Brooklyn,2018-03-01,We build plug & play e-bike tech,4730000,https://images.crunchbase.com/image/upload/t_cb-def

In [64]:
jobs_in_scope = jobs.loc[jobs['job_type'].isin(['executive', 'investor', 'advisor', 'board_member'])][['person_uuid', 'org_uuid', 'started_on', 'ended_on', 'job_title', 'job_type']]

jobs_investments = pd.concat([jobs_in_scope, investments_info.loc[investments_info['person_uuid'].isin(jobs_in_scope['person_uuid'])]], ignore_index = True)

jobs_investments = pd.merge(jobs_investments, organisations[['org_uuid', 'org_name', 'org_country_code', 'org_city', 'founded_on', 'short_description', 'total_funding_usd', 'org_logo_url']], how = 'left', on = 'org_uuid')

jobs_investments = pd.merge(jobs_investments, people[['person_uuid', 'person_name', 'person_logo_url']], how = 'left', on = 'person_uuid')

all_info = pd.merge(jobs_investments, acquisitons_ipos, how = 'left', on = 'org_uuid')

subsequent_exploded = pd.merge(all_info, all_info, on = 'person_uuid', suffixes = ('_target', '_subsequent'))

alumni_master = subsequent_exploded.loc[
    (subsequent_exploded['org_uuid_target'] != subsequent_exploded['org_uuid_subsequent']) # Remove entry of target org
    & (subsequent_exploded['job_type_target'] == 'executive') # Target org role must be executive
    & (subsequent_exploded['started_on_subsequent'] >= subsequent_exploded['started_on_target']) # Subsequrnt must be after target start date
    & (subsequent_exploded['started_on_target'] <= subsequent_exploded['founded_on_target'] + pd.DateOffset(years = years_founding_cutoff))
].rename(columns = {'person_name_target': 'person_name', 'job_type_subsequent': 'relation_type', 'person_logo_url_target': 'person_logo_url'}).drop(columns = ['person_name_subsequent', 'person_logo_url_subsequent'], axis = 1)

alumni_master = alumni_master.drop(columns = ['acquirer_uuid_target', 'exit_type_target', 'acquirer_name_target', 'exit_date_target', 'acquisition_type_target', 'exit_valuation_target', 'short_description_target', 'total_funding_usd_target', 'org_logo_url_target'], axis = 1)

# Prioritise which relation with same subsequent company should be kep
alumni_master['priority'] = alumni_master['relation_type'].map(relation_priority_map)
alumni_master = alumni_master.sort_values(by = 'priority')

# Drop the priority column as it's no longer needed
alumni_master = alumni_master.drop(columns = ['priority'], axis = 1)

# Define columns to exclude
exclude_columns = ['job_title_subsequent', 'relation_type', 'started_on_subsequent', 'ended_on_subsequent']

# Get all columns except for the ones to exclude
subset_columns = alumni_master.columns.difference(exclude_columns).tolist()

# Drop duplicates keeping the one with the highest priority
alumni_master = alumni_master.drop_duplicates(subset = subset_columns, keep = 'first')

alumni_master = alumni_master.dropna(subset = ['person_uuid', 'org_uuid_target', 'org_uuid_subsequent'])

alumni_master['record_uuid'] = [uuid.uuid4() for _ in range(len(alumni_master))] # Adding record_uuid as primary key in DB

alumni_master

,person_uuid,org_uuid_target,started_on_target,ended_on_target,job_title_target,job_type_target,org_name_target,org_country_code_target,org_city_target,founded_on_target,person_name,person_logo_url,org_uuid_subsequent,started_on_subsequent,ended_on_subsequent,job_title_subsequent,relation_type,org_name_subsequent,org_country_code_subsequent,org_city_subsequent,founded_on_subsequent,short_description_subsequent,total_funding_usd_subsequent,org_logo_url_subsequent,acquirer_uuid_subsequent,exit_type_subsequent,acquirer_name_subsequent,exit_date_subsequent,acquisition_type_subsequent,exit_valuation_subsequent,record_uuid
3,ed13cd36-fe2b-3707-197b-0c2d56e37a71,e1393508-30ea-8a36-3f96-dd3226033abd,2005-10-01,2014-06-01,Co-Founder and CEO,executive,Wetpaint,USA,New York,2005-06-01,Ben Elowitz,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180224/56303d2f4b99dd1dcc8abf17ba3fd8bf.jpg,cf253887-5eac-21a2-28d3-47db7311f7e9,2018-06-01,2019-03-01,Managing Director,executive,Madrona,USA,Seattle,1995-01-01,Madrona is a venture firm that invests in early- to late-stage companies in the Pacific Northwest and beyond.,<NA>,https://images.crunchbase.com/image/upload/t_cb-default-original/cpqilcg3lmakawavpmiv,NaN,NaN,NaN,NaT,NaN,<NA>,790cb469-56c0-4943-be21-f5973ae0a77d
6983412,9ebbbeb5-3164-adfd-d8cf-c0fd47f943ff,7425f8f0-d94e-435c-bc90-1c0a84545dae,2018-08-01,NaT,Co-Founder Executive Director,executive,Marion Square,USA,Charlotte,2016-01-01,Harvey Morrison,https://images.crunchbase.com/image/upload/t_cb-default-original/v1468456768/ijtcagvew4p1pki2basn.png,03cbab55-f97e-ef60-4b4b-22fda2786723,2019-06-01,NaT,Chief Revenue Officer,executive,Blue Cedar,USA,San Francisco,2016-01-01,Every App Secured. No Coding Required.,27000000,https://images.crunchbase.com/image/upload/t_cb-default-original/qaeu3p9nnhsvwlvjm5m4,NaN,NaN,NaN,NaT,NaN,<NA>,6b4ef23d-0d13-4f01-b802-64db2a4e3109
6983374,ac36e6f1-0b3c-2f12-6983-c10590d47cb0,c9721a72-c2fc-7286-d67e-99c33698d36b,2005-11-01,2011-07-01,Founder,executive,Euristix,GBR,London,2005-01-01,Frederic Nze,https://images.crunchbase.com/image/upload/t_cb-default-original/quumkfbxulgs1scukvg8,3134a8af-179d-4687-9acb-5d70f252a32e,2006-08-01,2021-07-01,Founder,executive,Oakam,GBR,Croydon,2006-01-01,Oakam is a digital micro-lender that provides credit scoring and loans to individuals as per the requirement.,<NA>,https://images.crunchbase.com/image/upload/t_cb-default-original/yhsnsfd3iqjccc1takek,NaN,NaN,NaN,NaT,NaN,<NA>,3b127299-2332-4c46-8c8a-062a1c3fa9d7
6983373,ac36e6f1-0b3c-2f12-6983-c10590d47cb0,c9721a72-c2fc-7286-d67e-99c33698d36b,2005-11-01,2011-07-01,Founder,executive,Euristix,GBR,London,2005-01-01,Frederic Nze,https://images.crunchbase.com/image/upload/t_cb-default-original/quumkfbxulgs1scukvg8,bf98342e-33fb-41e6-9fd3-bc9a3234e108,2019-03-01,NaT,Founder,executive,Akrod,GBR,London,2019-03-04,"Akrod drives financial inclusion through the use of mobile technology and alternative data to innovate, scale, and transform microfinance.",17065236,https://images.crunchbase.com/image/upload/t_cb-default-original/kufzc8cyqflxxlndekdh,NaN,NaN,NaN,NaT,NaN,<NA>,34f558b0-dce5-402d-9b5b-612d4c1ef072
6983370,ac36e6f1-0b3c-2f12-6983-c10590d47cb0,c9721a72-c2fc-7286-d67e-99c33698d36b,2005-11-01,2011-07-01,Founder,executive,Euristix,GBR,London,2005-01-01,Frederic Nze,https://images.crunchbase.com/image/upload/t_cb-default-original/quumkfbxulgs1scukvg8,fd794613-994e-d656-1b03-56d6298dfdd0,2006-08-01,2022-08-01,CEO and Founder,executive,Oakam,NaN,NaN,2006-08-01,Ceased Trading,54451721,https://images.crunchbase.com/image/upload/t_cb-default-original/tn3id1n9g8lynxsrk9k9,bf98342e-33fb-41e6-9fd3-bc9a3234e108,acquisition,Akrod,2019-03-01,NaN,<NA>,f4854619-c0a7-4393-8c4a-4eae98d8dddb
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8239333,b4229aad-7c03-4bdb-90f8-30c514e4973f,2f0d6022-25e9-4f42-bca3-771d44323d0a,2017-06-05,NaT,Founder and CEO,

# Support functions

In [65]:
def export(df, org_uuid, name):
    df.to_json('data/output/' + name + '_' + org_uuid + '.json', orient = 'records')

In [66]:
# Hash function to derive a new uuid from the Crunchbase uuid
def derive_new_uuid(original_uuid):
    original_uuid_str = str(original_uuid)
    hash_object = hashlib.sha256(original_uuid_str.encode())
    hash_digest = hash_object.hexdigest()
    derived_uuid = uuid.UUID(hash_digest[:32])
    return derived_uuid

# Dataframe functions

## target_org - Information about the target organisation

In [67]:
def func_target_org(org_uuid):
    target_org = organisations.loc[organisations['org_uuid'] == org_uuid][['org_uuid', 'org_name', 'org_country_code', 'org_region', 'org_city', 'short_description', 'total_funding_usd', 'founded_on', 'org_logo_url']]

    # Add exit info
    target_org = pd.merge(target_org, acquisitons_ipos, how = 'left', on = 'org_uuid')

    result = target_org.rename(columns={
        'org_uuid': 'orgUuid',
        'org_name': 'orgName',
        'org_country_code': 'orgCountryCode',
        'org_region': 'orgRegion',
        'org_city': 'orgCity',
        'short_description': 'shortDescription',
        'total_funding_usd': 'totalFundingUsd',
        'founded_on': 'foundedOn',
        'org_logo_url': 'orgLogoUrl',
        'exit_type': 'exitType',
        'exit_date': 'exitDate',
        'exit_valuation': 'exitValuation',
        'acquirer_uuid': 'acquirerUuid',
        'acquirer_name': 'acquirerName',
        'acquisition_type': 'acquisitionType',
    }).drop_duplicates()
    
    return result

## alumni_info - Information about target organisation alumnis

In [68]:
def func_alumni_info(org_uuid):
    alumni_info = alumni_master.loc[alumni_master['org_uuid_target'] == org_uuid][['person_uuid', 'person_name', 'job_title_target', 'job_type_target', 'started_on_target', 'ended_on_target', 'person_logo_url']]
    
    result = alumni_info.drop_duplicates(subset = ['person_uuid']).rename(columns={
        'person_uuid': 'personUuid',
        'person_name': 'personName',
        'job_title_target': 'jobTitle',
        'job_type_target': 'jobType',
        'started_on_target': 'startedOn',
        'ended_on_target': 'endedOn',
        'person_logo_url': 'personLogoUrl',
    }).drop_duplicates()
    
    return result

## subsequent_orgs - Organisations the alumni of the target organisation are associated with

In [69]:
def func_subsequent_orgs_info(org_uuid):
    subsequent_orgs_info = alumni_master.loc[alumni_master['org_uuid_target'] == org_uuid][['org_uuid_subsequent', 'org_name_subsequent', 'org_country_code_subsequent', 'org_city_subsequent', 'short_description_subsequent', 'total_funding_usd_subsequent', 'founded_on_subsequent', 'org_logo_url_subsequent', 'exit_type_subsequent', 'exit_date_subsequent', 'exit_valuation_subsequent', 'acquirer_uuid_subsequent', 'acquirer_name_subsequent', 'acquisition_type_subsequent']]
    
    result = subsequent_orgs_info.rename(columns={
        'org_uuid_subsequent': 'orgUuid',
        'org_name_subsequent': 'orgName',
        'org_country_code_subsequent': 'orgCountryCode',
        'org_city_subsequent': 'orgCity',
        'short_description_subsequent': 'shortDescription',
        'total_funding_usd_subsequent': 'totalFundingUsd',
        'founded_on_subsequent': 'foundedOn',
        'org_logo_url_subsequent': 'orgLogoUrl',
        'exit_type_subsequent': 'exitType',
        'exit_date_subsequent': 'exitDate',
        'exit_valuation_subsequent': 'exitValuation',
        'acquirer_uuid_subsequent': 'acquirerUuid',
        'acquirer_name_subsequent': 'acquirerName',
        'acquisition_type_subsequent': 'acquisitionType',
    }).drop_duplicates()
    
    return result

## relations - Relationships between target org people and subsequent org

In [70]:
def func_relations(org_uuid):
    relations = alumni_master.loc[alumni_master['org_uuid_target'] == org_uuid][['person_uuid', 'org_uuid_subsequent', 'relation_type', 'job_title_subsequent', 'org_logo_url_subsequent']]
    
    result = relations.rename(columns={
        'person_uuid': 'personUuid',
        'org_uuid_subsequent': 'orgUuid',
        'relation_type': 'relationType',
        'job_title_subsequent': 'jobTitleSubsequent',
        'org_logo_url_subsequent': 'orgLogoUrl',
    }).drop_duplicates()
    
    return result

# Export

In [71]:
subsequent_orgs_count = (
    alumni_master
        .groupby('org_uuid_target', as_index = False)
        .agg(
            {
                'org_uuid_subsequent': 'nunique'
            },
            skipna = True
        )
).sort_values(by = 'org_uuid_subsequent', ascending = False)

subsequent_orgs_count = pd.merge(subsequent_orgs_count, organisations, how = 'left', left_on = 'org_uuid_target', right_on = 'org_uuid')[['org_uuid_target', 'org_name', 'org_uuid_subsequent', 'org_country_code']]

subsequent_orgs_count.head(50)

,org_uuid_target,org_name,org_uuid_subsequent,org_country_code
0,2c23e7ad-4441-5320-fdf9-5aa7007fec0b,PwC,754,GBR
1,96ab87ca-00b5-2ebc-f218-86262954e320,PayPal,584,USA
2,ca51c248-b907-b20b-bd0c-0d0cbd07f615,GovPredict,510,USA
3,df662812-7f97-0b43-9d3e-12f64f504fbb,Meta,509,USA
4,93f26870-e9c2-4bc8-aa6d-ecf3f73c1fb1,Catalina Crunch,464,USA
5,1bcd5c8d-5534-4f7c-8702-72d90ca9a0cb,Atom Finance,446,USA
6,5da6106f-0d27-0d37-e9d7-dcfeccc1f709,X (formerly Twitter),395,USA
7,9e4980a4-7be8-4eae-a994-0c2e114d9181,Lido,366,USA
8,a4af56b9-4f9c-4f0e-ae9e-4d1dffe72d44,Pareto Holdings,366,USA
9,297df5af-da41-a709-7df2-7e7f04e53454,Zynga,346,USA


In [72]:
subsequent_orgs_count.loc[subsequent_orgs_count['org_name'] == 'Google']

,org_uuid_target,org_name,org_uuid_subsequent,org_country_code
54,6acfa7da-1dbd-936e-d985-cf07a1b27711,Google,157,USA


In [73]:
subsequent_orgs_count.loc[subsequent_orgs_count['org_country_code'] == 'SWE'].head(20)

,org_uuid_target,org_name,org_uuid_subsequent,org_country_code
198,022417b5-4980-6c54-0f3c-6736bbbb1a5e,Spotify,88,SWE
208,919c3bde-3f44-b0e4-1b41-204d3475e877,Nordic Makers,87,SWE
560,ca5d38b8-9c67-8938-2f93-dcb75d41d1d9,EQT Ventures,49,SWE
731,7b74b50c-4468-b7ec-2ff9-bf3286a399c9,Kry,44,SWE
908,6093de34-5382-f34c-a207-efa92470048b,iZettle,39,SWE
1015,42f47614-0ce4-fd2c-767d-e29c5fdff472,Stardoll,36,SWE
1186,4b68dde5-7099-deac-bb55-7eebd0d334ae,Brisk,33,SWE
1446,4ede174d-3254-8602-e977-d9c0bfe34433,Creandum,30,SWE
1449,d39bb6dc-582d-4878-a224-005497e03766,VOI Technology,30,SWE
1490,51d94123-97d9-be50-33a7-c1ee3f2bee0f,Wrapp,30,SWE


In [74]:
options_json = '''
[
    {"value": "96ab87ca-00b5-2ebc-f218-86262954e320", "label": "PayPal"},
    {"value": "a367b036-5952-5435-7541-ad7ee8869e24", "label": "Tesla"},
    {"value": "df662812-7f97-0b43-9d3e-12f64f504fbb", "label": "Meta"},
    {"value": "022417b5-4980-6c54-0f3c-6736bbbb1a5e", "label": "Spotify"},
    {"value": "2cc3a5de-2303-aa00-cd1a-50bd96420392", "label": "Klarna"},
    {"value": "34035c51-8f16-4836-8f02-103392479a92", "label": "Northvolt"},
    {"value": "7b74b50c-4468-b7ec-2ff9-bf3286a399c9", "label": "Kry"},
    {"value": "d39bb6dc-582d-4878-a224-005497e03766", "label": "Voi"},
    {"value": "439d3478-40fa-e6bc-9b71-f1bfa8296f52", "label": "Epidemic Sound"},
    {"value": "6093de34-5382-f34c-a207-efa92470048b", "label": "iZettle"},
    {"value": "fd80725f-53fc-7009-9878-aeecf1e9ffbb", "label": "Microsoft"},
    {"value": "6acfa7da-1dbd-936e-d985-cf07a1b27711", "label": "Google"},
    {"value": "eef9eab2-4c50-f0a3-12b8-ce721fa2cc81", "label": "YouTube"},
    {"value": "cc68526b-b2d7-4f7f-cfa7-d93b23716027", "label": "Netscape"},
    {"value": "2c23e7ad-4441-5320-fdf9-5aa7007fec0b", "label": "PwC"},
    {"value": "b3a7efa7-e3d3-2d4c-962c-b9fdb1da496f", "label": "Opsware"},
    {"value": "1eb37109-3b93-01a9-177f-fee2cb1bfcdc", "label": "Uber"},
    {"value": "9a0e860a-7743-28e2-05c0-2b08646d0fe1", "label": "Skype"},
    {"value": "988b8953-a085-8de4-babb-fae5bb895761", "label": "Foodpanda"},
    {"value": "643d60b4-bfa8-ee61-3316-5af0d8325f33", "label": "Excite"},
    {"value": "0d5171b3-68b3-37c3-cb50-8cd8ccb8930b", "label": "Accenture"},
    {"value": "4b724583-6013-30ca-88b0-35e5c6e71ad5", "label": "Epinions"},
    {"value": "297df5af-da41-a709-7df2-7e7f04e53454", "label": "Zynga"},
    {"value": "e56b0ceb-bb30-bbec-805e-d5dc7412dcb1", "label": "eBay"},
    {"value": "a6b663d3-586a-ad04-924c-1ca87763b2fb", "label": "FreeCharge"},
    {"value": "08639f0b-56fd-997f-5c9d-0ca3b5d9672b", "label": "Justin.tv"},
    {"value": "1966bb69-4bc9-1077-c181-fe3d67509160", "label": "Pobts"},
    {"value": "86da6213-5b43-6419-4047-472102ccf66f", "label": "LinkedIn"},
    {"value": "f5c477fa-6e8c-3d64-4f2d-3603e5cc3340", "label": "Salesforce"},
    {"value": "31fc3998-a1f1-2e5b-6358-f8d068fa9f71", "label": "Delivery Hero"},
    {"value": "6f7ea63a-f820-733a-702b-82e75d0ac15f", "label": "Wimdu"},
    {"value": "70756e51-3859-a1ae-8edb-d7eeaf5ed342", "label": "Xing"},
    {"value": "ffd3825c-ac83-fa1e-0d5b-538c08904cb6", "label": "Wunderlist"},
    {"value": "04f52814-77e2-b2c9-2d9a-4299dbb62455", "label": "Rocket Internet"},
    {"value": "dd92791e-2081-c3fc-f0a1-f1e2633dadcd", "label": "dreamfab"},
    {"value": "a6f5492b-1215-6c03-205f-1efb9913f2e2", "label": "Infineon Technologies"},
    {"value": "988b8953-a085-8de4-babb-fae5bb895761", "label": "Foodpanda"},
    {"value": "36027bbe-4f12-f224-f90f-2ff31c0294af", "label": "Team Global"},
    {"value": "0a9eae4d-a269-646c-0cad-cf5b559b74b6", "label": "Zalando"},
    {"value": "251270ba-b3b8-6135-ed82-6657e1c8b046", "label": "N26"},
    {"value": "e7e3805e-3997-6000-a068-b667c249dbc5", "label": "Alcatel-Lucent"},
    {"value": "24aeb98f-6cd3-f55b-1575-72eec5ba2403", "label": "Amen.fr"},
    {"value": "1e8e57cf-9c81-a082-7512-723473e81dbb", "label": "Veepee"},
    {"value": "1bbcec66-242d-4650-b582-47305d6f3d87", "label": "folk"},
    {"value": "b485a6e1-2cc8-3388-d949-240863fdccf8", "label": "KelDoc"}
]
'''

options = json.loads(options_json)

filtered_orgs = organisations[organisations['total_funding_usd'] > 30000000]

existing_uuids = {option['value'] for option in options}

# Add new companies to the options_list if they are not already present
for _, org in filtered_orgs.iterrows():
    if org['org_uuid'] not in existing_uuids:
        # Creating a new value for the new company, typically this should be a unique ID
        new_option = {"value": org['org_uuid'], "label": org['org_name']}
        options.append(new_option)

In [75]:
options_json_production = '''
[
    {"value": "df662812-7f97-0b43-9d3e-12f64f504fbb", "label": "Meta"}
]
'''
options_json_production = '''
[
    {"value": "6acfa7da-1dbd-936e-d985-cf07a1b27711", "label": "Google"},
    {"value": "eef9eab2-4c50-f0a3-12b8-ce721fa2cc81", "label": "YouTube"},
    {"value": "b3a7efa7-e3d3-2d4c-962c-b9fdb1da496f", "label": "Opsware"},
    {"value": "08639f0b-56fd-997f-5c9d-0ca3b5d9672b", "label": "Justin.tv"},
    {"value": "5da6106f-0d27-0d37-e9d7-dcfeccc1f709", "label": "X (formerly Twitter)"},
    {"value": "297df5af-da41-a709-7df2-7e7f04e53454", "label": "Zynga"}
]
'''

options_production = json.loads(options_json_production)

## Full options list export (one file for DB)

In [76]:
values = [company["value"] for company in options_production]

In [77]:
target_org = organisations.loc[organisations['org_uuid'].isin(values)][['org_uuid', 'org_name', 'org_country_code', 'org_region', 'org_city', 'short_description', 'total_funding_usd', 'founded_on', 'org_logo_url']]

# Add exit info
target_org = pd.merge(target_org, acquisitons_ipos, how = 'left', on = 'org_uuid')

# Change to my own uuids
target_org = target_org.assign(org_uuid = target_org['org_uuid'].apply(derive_new_uuid))
target_org = target_org.assign(acquirer_uuid = target_org['acquirer_uuid'].apply(derive_new_uuid))

result = target_org.rename(columns={
    'org_uuid': 'orgUuid',
    'org_name': 'orgName',
    'org_country_code': 'orgCountryCode',
    'org_region': 'orgRegion',
    'org_city': 'orgCity',
    'short_description': 'shortDescription',
    'total_funding_usd': 'totalFundingUsd',
    'founded_on': 'foundedOn',
    'org_logo_url': 'orgLogoUrl',
    'exit_type': 'exitType',
    'exit_date': 'exitDate',
    'exit_valuation': 'exitValuation',
    'acquirer_uuid': 'acquirerUuid',
    'acquirer_name': 'acquirerName',
    'acquisition_type': 'acquisitionType',
}).drop_duplicates()

target_org.to_csv('data/output/' + 'targetOrgs' + '_' + 'full' + '.csv', index=False)

In [78]:
columns = [
    'record_uuid',
    'person_uuid',
    'org_uuid_target',
    'started_on_target',
    'ended_on_target',
    'job_title_target',
    'job_type_target',
    'org_name_target',
    'org_country_code_target',
    'org_city_target',
    'founded_on_target',
    'person_name',
    'person_logo_url',
    'org_uuid_subsequent',
    'started_on_subsequent',
    'ended_on_subsequent',
    'job_title_subsequent',
    'relation_type',
    'org_name_subsequent',
    'org_country_code_subsequent',
    'org_city_subsequent',
    'founded_on_subsequent',
    'short_description_subsequent',
    'total_funding_usd_subsequent',
    'org_logo_url_subsequent',
    'acquirer_uuid_subsequent',
    'exit_type_subsequent',
    'acquirer_name_subsequent',
    'exit_date_subsequent',
    'acquisition_type_subsequent',
    'exit_valuation_subsequent'
]

alumni_master_output = alumni_master.loc[alumni_master['org_uuid_target'].isin(values)][columns]

# Change to my own uuids
alumni_master_output = alumni_master_output.assign(person_uuid = alumni_master['person_uuid'].apply(derive_new_uuid))
alumni_master_output = alumni_master_output.assign(org_uuid_target = alumni_master['org_uuid_target'].apply(derive_new_uuid))
alumni_master_output = alumni_master_output.assign(org_uuid_subsequent = alumni_master['org_uuid_subsequent'].apply(derive_new_uuid))
alumni_master_output = alumni_master_output.assign(acquirer_uuid_subsequent = alumni_master['acquirer_uuid_subsequent'].apply(derive_new_uuid))

result = alumni_master_output.rename(columns={
    'record_uuid': 'recordUuid',
    'person_uuid': 'personUuid',
    'org_uuid_target': 'orgUuidTarget',
    'started_on_target': 'startedOnTarget',
    'ended_on_target': 'endedOnTarget',
    'job_title_target': 'jobTitleTarget',
    'job_type_target': 'jobTypeTarget',
    'org_name_target': 'orgNameTarget',
    'org_country_code_target': 'orgCountryCodeTarget',
    'org_city_target': 'orgCityTarget',
    'founded_on_target': 'foundedOnTarget',
    'person_name': 'personName',
    'person_logo_url': 'personLogoUrl',
    'org_uuid_subsequent': 'orgUuidSubsequent',
    'started_on_subsequent': 'startedOnSubsequent',
    'ended_on_subsequent': 'endedOnSubsequent',
    'job_title_subsequent': 'jobTitleSubsequent',
    'relation_type': 'relationType',
    'org_name_subsequent': 'orgNameSubsequent',
    'org_country_code_subsequent': 'orgCountryCodeSubsequent',
    'org_city_subsequent': 'orgCitySubsequent',
    'founded_on_subsequent': 'foundedOnSubsequent',
    'short_description_subsequent': 'shortDescriptionSubsequent',
    'total_funding_usd_subsequent': 'totalFundingUsdSubsequent',
    'org_logo_url_subsequent': 'orgLogoUrlSubsequent',
    'acquirer_uuid_subsequent': 'acquirerUuidSubsequent',
    'exit_type_subsequent': 'exitTypeSubsequent',
    'acquirer_name_subsequent': 'acquirerNameSubsequent',
    'exit_date_subsequent': 'exitDateSubsequent',
    'acquisition_type_subsequent': 'acquisitionTypeSubsequent',
    'exit_valuation_subsequent': 'exitValuationSubsequent'
}).drop_duplicates()

alumni_master_output.to_csv('data/output/' + 'alumni_master' + '_' + 'full' + '.csv', index=False)

## Output of subsequent companies

In [34]:
columns = [
    'record_uuid',
    'person_uuid',
    'org_uuid_target',
    'started_on_target',
    'ended_on_target',
    'job_title_target',
    'job_type_target',
    'org_name_target',
    'org_country_code_target',
    'org_city_target',
    'founded_on_target',
    'person_name',
    'person_logo_url',
    'org_uuid_subsequent',
    'started_on_subsequent',
    'ended_on_subsequent',
    'job_title_subsequent',
    'relation_type',
    'org_name_subsequent',
    'org_country_code_subsequent',
    'org_city_subsequent',
    'founded_on_subsequent',
    'short_description_subsequent',
    'total_funding_usd_subsequent',
    'org_logo_url_subsequent',
    'acquirer_uuid_subsequent',
    'exit_type_subsequent',
    'acquirer_name_subsequent',
    'exit_date_subsequent',
    'acquisition_type_subsequent',
    'exit_valuation_subsequent'
]

columns_subsequent = [
    'org_uuid_subsequent',
    'started_on_subsequent',
    'ended_on_subsequent',
    'job_title_subsequent',
    'relation_type',
    'org_name_subsequent',
    'org_country_code_subsequent',
    'org_city_subsequent',
    'founded_on_subsequent',
    'short_description_subsequent',
    'total_funding_usd_subsequent',
    'org_logo_url_subsequent',
    'acquirer_uuid_subsequent',
    'exit_type_subsequent',
    'acquirer_name_subsequent',
    'exit_date_subsequent',
    'acquisition_type_subsequent',
    'exit_valuation_subsequent'
]

alumni_master = alumni_master.loc[alumni_master['org_uuid_target'].isin(values)][columns]

# Change to my own uuids
alumni_master = alumni_master.assign(person_uuid = alumni_master['person_uuid'].apply(derive_new_uuid))
alumni_master = alumni_master.assign(org_uuid_target = alumni_master['org_uuid_target'].apply(derive_new_uuid))
alumni_master = alumni_master.assign(org_uuid_subsequent = alumni_master['org_uuid_subsequent'].apply(derive_new_uuid))
alumni_master = alumni_master.assign(acquirer_uuid_subsequent = alumni_master['acquirer_uuid_subsequent'].apply(derive_new_uuid))

subsequent = alumni_master.drop_duplicates(subset = ['org_uuid_subsequent'])[columns_subsequent]

subsequent.to_csv('data/output/' + 'subsequent' + '_' + 'full' + '.csv', index=False)

## Individual files export - NOT USED

In [ ]:
for item in options:
    target_org_uuid = item['value']
    print(target_org_uuid)
    export(func_target_org(target_org_uuid), target_org_uuid, 'targetOrg')
    export(func_alumni_info(target_org_uuid), target_org_uuid, 'alumniInfo')
    export(func_subsequent_orgs_info(target_org_uuid), target_org_uuid, 'subsequentOrgsInfo')
    export(func_relations(target_org_uuid), target_org_uuid, 'relations')

In [ ]:
target_org_uuid = 'ada0db7e-2854-4d21-8e6a-eee86e978182'

export(func_target_org(target_org_uuid), target_org_uuid, 'targetOrg')
export(func_alumni_info(target_org_uuid), target_org_uuid, 'alumniInfo')
export(func_subsequent_orgs_info(target_org_uuid), target_org_uuid, 'subsequentOrgsInfo')
export(func_relations(target_org_uuid), target_org_uuid, 'relations')

In [81]:
alumni_master

,person_uuid,org_uuid_target,started_on_target,ended_on_target,job_title_target,job_type_target,org_name_target,org_country_code_target,org_city_target,founded_on_target,person_name,person_logo_url,org_uuid_subsequent,started_on_subsequent,ended_on_subsequent,job_title_subsequent,relation_type,org_name_subsequent,org_country_code_subsequent,org_city_subsequent,founded_on_subsequent,short_description_subsequent,total_funding_usd_subsequent,org_logo_url_subsequent,acquirer_uuid_subsequent,exit_type_subsequent,acquirer_name_subsequent,exit_date_subsequent,acquisition_type_subsequent,exit_valuation_subsequent,record_uuid
3,ed13cd36-fe2b-3707-197b-0c2d56e37a71,e1393508-30ea-8a36-3f96-dd3226033abd,2005-10-01,2014-06-01,Co-Founder and CEO,executive,Wetpaint,USA,New York,2005-06-01,Ben Elowitz,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180224/56303d2f4b99dd1dcc8abf17ba3fd8bf.jpg,cf253887-5eac-21a2-28d3-47db7311f7e9,2018-06-01,2019-03-01,Managing Director,executive,Madrona,USA,Seattle,1995-01-01,Madrona is a venture firm that invests in early- to late-stage companies in the Pacific Northwest and beyond.,<NA>,https://images.crunchbase.com/image/upload/t_cb-default-original/cpqilcg3lmakawavpmiv,NaN,NaN,NaN,NaT,NaN,<NA>,790cb469-56c0-4943-be21-f5973ae0a77d
6983412,9ebbbeb5-3164-adfd-d8cf-c0fd47f943ff,7425f8f0-d94e-435c-bc90-1c0a84545dae,2018-08-01,NaT,Co-Founder Executive Director,executive,Marion Square,USA,Charlotte,2016-01-01,Harvey Morrison,https://images.crunchbase.com/image/upload/t_cb-default-original/v1468456768/ijtcagvew4p1pki2basn.png,03cbab55-f97e-ef60-4b4b-22fda2786723,2019-06-01,NaT,Chief Revenue Officer,executive,Blue Cedar,USA,San Francisco,2016-01-01,Every App Secured. No Coding Required.,27000000,https://images.crunchbase.com/image/upload/t_cb-default-original/qaeu3p9nnhsvwlvjm5m4,NaN,NaN,NaN,NaT,NaN,<NA>,6b4ef23d-0d13-4f01-b802-64db2a4e3109
6983374,ac36e6f1-0b3c-2f12-6983-c10590d47cb0,c9721a72-c2fc-7286-d67e-99c33698d36b,2005-11-01,2011-07-01,Founder,executive,Euristix,GBR,London,2005-01-01,Frederic Nze,https://images.crunchbase.com/image/upload/t_cb-default-original/quumkfbxulgs1scukvg8,3134a8af-179d-4687-9acb-5d70f252a32e,2006-08-01,2021-07-01,Founder,executive,Oakam,GBR,Croydon,2006-01-01,Oakam is a digital micro-lender that provides credit scoring and loans to individuals as per the requirement.,<NA>,https://images.crunchbase.com/image/upload/t_cb-default-original/yhsnsfd3iqjccc1takek,NaN,NaN,NaN,NaT,NaN,<NA>,3b127299-2332-4c46-8c8a-062a1c3fa9d7
6983373,ac36e6f1-0b3c-2f12-6983-c10590d47cb0,c9721a72-c2fc-7286-d67e-99c33698d36b,2005-11-01,2011-07-01,Founder,executive,Euristix,GBR,London,2005-01-01,Frederic Nze,https://images.crunchbase.com/image/upload/t_cb-default-original/quumkfbxulgs1scukvg8,bf98342e-33fb-41e6-9fd3-bc9a3234e108,2019-03-01,NaT,Founder,executive,Akrod,GBR,London,2019-03-04,"Akrod drives financial inclusion through the use of mobile technology and alternative data to innovate, scale, and transform microfinance.",17065236,https://images.crunchbase.com/image/upload/t_cb-default-original/kufzc8cyqflxxlndekdh,NaN,NaN,NaN,NaT,NaN,<NA>,34f558b0-dce5-402d-9b5b-612d4c1ef072
6983370,ac36e6f1-0b3c-2f12-6983-c10590d47cb0,c9721a72-c2fc-7286-d67e-99c33698d36b,2005-11-01,2011-07-01,Founder,executive,Euristix,GBR,London,2005-01-01,Frederic Nze,https://images.crunchbase.com/image/upload/t_cb-default-original/quumkfbxulgs1scukvg8,fd794613-994e-d656-1b03-56d6298dfdd0,2006-08-01,2022-08-01,CEO and Founder,executive,Oakam,NaN,NaN,2006-08-01,Ceased Trading,54451721,https://images.crunchbase.com/image/upload/t_cb-default-original/tn3id1n9g8lynxsrk9k9,bf98342e-33fb-41e6-9fd3-bc9a3234e108,acquisition,Akrod,2019-03-01,NaN,<NA>,f4854619-c0a7-4393-8c4a-4eae98d8dddb
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8239333,b4229aad-7c03-4bdb-90f8-30c514e4973f,2f0d6022-25e9-4f42-bca3-771d44323d0a,2017-06-05,NaT,Founder and CEO,